# 📊 35 · Statistical genetics at scale: mixed models, fine-mapping, polygenic scores and Mendelian randomisation
### From the Module 06 GWAS to biobank practice: population structure and genetic relationship matrices, SNP heritability with linear mixed models, Bayesian fine-mapping with credible sets, polygenic scores and why they travel badly between populations, Mendelian randomisation and LD-score regression

**BiologyToML** · Level 2 · Research frontier · Part H — Genomes, proteins and molecules · Module 35 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/35_statistical_genetics.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, Matplotlib, scikit-learn (lasso only) |
| **Prerequisites** | Level 1, especially Modules 05 (likelihood), 06 (GWAS), 07 (ridge and lasso), 14 (PCA) and 21 (Bayesian inference); comfortable with NumPy and matrix algebra |
| **Estimated time** | 185 min (reading + running + the ✏️ exercises) |
| **Builds on** | [05 · Likelihood, MLE and MAP](05_likelihood_mle_map.ipynb), [06 · Linear regression and GWAS](06_linear_regression_gwas.ipynb), [07 · Ridge, lasso and sparse genetic effects](07_ridge_lasso_sparse_regression.ipynb), [14 · PCA: genes mirror geography](14_pca_dimensionality_reduction.ipynb), [21 · Bayesian learning and MCMC](21_bayesian_learning_mcmc.ipynb), [23 · Model selection](23_model_selection.ipynb) |
| **Research frontier covered** | biobank-scale GWAS, population structure and linear mixed models, SNP heritability, statistical fine-mapping, polygenic scores and their portability, Mendelian randomisation — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED**, so that every method can be checked against a known truth: genotypes under
> Hardy–Weinberg equilibrium with optional population structure (`simulate_genotypes`, Balding–Nichols $F_{ST}$),
> genotypes with realistic linkage-disequilibrium blocks (`simulate_ld_genotypes`), quantitative traits with known
> causal SNPs and heritability (`simulate_quantitative_trait`), and small in-notebook simulators for Mendelian
> randomisation and LD-score regression. The hidden truth includes the causal SNPs, their effects, $h^2$, the
> population labels and the true causal effect of the exposure. Real biobanks have millions of imputed variants,
> relatives, binary diseases, batch effects and consent constraints — Section 8 explains what changes.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Diagnose** a genome-wide scan with Manhattan and QQ plots and the genomic-inflation factor $\lambda_{GC}$, and **show** how population structure creates false hits and how genotype principal components remove them.
2. **Build** a genetic relationship matrix (GRM), **explain** what an entry $K_{ij}$ means, and **fit** a linear mixed model by (restricted) maximum likelihood with the eigen-rotation trick of EMMA/FaST-LMM to estimate SNP heritability with a confidence interval — then **verify** it against the true $h^2$ over replicate studies.
3. **Connect** the mixed model to ridge regression (BLUP = MAP with a Gaussian prior, Module 07) and **use** it as an association test that controls confounding.
4. **Fine-map** an LD region: compute Bayes factors, posterior inclusion probabilities and 95 % credible sets, **implement** a simplified SuSiE for several causal variants, and **check** credible-set calibration by simulation.
5. **Construct** polygenic scores (clumping + thresholding, ridge/BLUP, lasso), **compare** them with the heritability ceiling, and **demonstrate** why their accuracy drops in a population with different LD.
6. **Estimate** a causal effect by Mendelian randomisation (Wald ratio, inverse-variance weighting, MR-Egger, weighted median), **state** the three instrumental-variable assumptions and **provoke** their violation.
7. **Explain** LD-score regression — why the intercept measures confounding and the slope polygenicity — and **name** the production tools, resources and ethical questions of biobank genetics.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, course toolkit | 5 min |
| 1 · From Module 06 to biobank GWAS | covariate-adjusted scans, Manhattan and QQ plots, $\lambda_{GC}$ | 15 min |
| 2 · Population structure | three populations, spurious hits, PCs from the GRM | 20 min |
| 3 · Linear mixed models and heritability | GRM, FaST-LMM likelihood, $\hat h^2$ ± CI, LMM association | 30 min |
| 4 · LD and fine-mapping | towers of association, Bayes factors, credible sets, SuSiE-lite | 30 min |
| 5 · Polygenic scores | C+T vs ridge/BLUP vs lasso; portability across populations | 25 min |
| 6 · Mendelian randomisation | instruments, Wald ratio, IVW, MR-Egger, weighted median | 20 min |
| 7 · Summary statistics: LD-score regression | slope = polygenicity, intercept = confounding | 15 min |
| 8 · 🧭 From toy to real data | biobanks, production tools, what goes wrong, ethics | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 15 min |

## 🧫 The biological question

**UK Biobank** recruited about half a million volunteers aged 40–69, measured hundreds of traits and genotyped every
participant at hundreds of thousands of variants, later imputed to tens of millions (Bycroft *et al.* 2018). FinnGen,
Biobank Japan, the Million Veteran Program and *All of Us* followed. With data on this scale, four questions that a
single-laboratory study could never answer became routine:

1. **Which variants cause disease risk?** A genome-wide scan (Module 06) finds *regions*, but neighbouring variants are
   inherited together (linkage disequilibrium), so the most significant SNP is often not the causal one.
2. **How heritable is a trait, and where is the heritability?** Twin studies say adult height is roughly 80 % heritable,
   yet the first GWAS hits explained only a few percent — the "missing heritability". Yang *et al.* (2010) showed with
   a **linear mixed model** that all common SNPs *together* explain about 45 % of the variance in height: the
   heritability was hiding in thousands of variants too weak to pass a significance threshold one by one.
3. **Can we predict risk from DNA — fairly?** Polygenic scores add up thousands of small effects. They work best in
   people whose ancestry matches the (mostly European-ancestry) discovery cohorts and lose much of their accuracy
   elsewhere (Martin *et al.* 2019) — a scientific *and* an equity problem.
4. **Does a risk factor cause a disease?** Randomised trials are slow and expensive. Because alleles are shuffled at
   conception, genetic variants that raise LDL cholesterol act like a lifelong, natural randomised trial
   (**Mendelian randomisation**, Davey Smith & Ebrahim 2003). Such studies support a causal role of LDL in coronary
   heart disease, while variants that raise HDL cholesterol did not lower heart-attack risk (Voight *et al.* 2012) —
   consistent with the disappointing trials of HDL-raising drugs.

In this notebook we build a **mini-biobank in silico** — a few thousand simulated people with known causal variants,
known heritability and known causal effects — and implement, from scratch, the core algorithm behind each of these
questions. Every result is checked against the truth we planted. Why simulate? Because only then can we *know* whether
a method found the causal SNP, estimated $h^2$ correctly or recovered the causal effect; because the repository stays
small and the notebook runs anywhere; and because we can switch confounders on and off at will. What simulations
cannot teach — messy real data — is the subject of Section 8.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats
from sklearn.linear_model import Lasso

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We use the three genotype/trait simulators (Hardy–Weinberg genotypes with optional population structure, LD-block genotypes, polygenic traits with known causal SNPs), `standardize`, the Benjamini–Hochberg FDR correction and two plotting helpers. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import os
import numpy as np
import matplotlib.pyplot as plt

def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


def simulate_quantitative_trait(rng: np.random.Generator, G: np.ndarray, n_causal: int = 5,
                                h2: float = 0.5, causal=None):
    """Simulate a quantitative phenotype y = G·β + ε with ``n_causal`` causal SNPs.

    The genetic part is scaled so that it explains a fraction ``h2`` (the heritability) of
    the phenotypic variance.  Returns ``(y, beta, causal_idx)``.
    """
    N, J = G.shape
    causal_idx = np.sort(rng.choice(J, n_causal, replace=False) if causal is None else np.asarray(causal))
    beta = np.zeros(J)
    beta[causal_idx] = rng.choice([-1, 1], len(causal_idx)) * rng.uniform(0.5, 1.0, len(causal_idx))
    g = G @ beta
    scale = np.sqrt(h2 / g.var()) if g.var() > 0 else 0.0
    beta *= scale
    y = G @ beta + rng.normal(0, np.sqrt(1 - h2), N)
    return y, beta, causal_idx


def simulate_ld_genotypes(rng: np.random.Generator, n_samples: int = 2000, n_snps: int = 400,
                          n_founders: int = 8, block_length: int = 40, mutation: float = 0.002):
    """Simulate 0/1/2 genotypes with realistic linkage disequilibrium (LD blocks).

    The genome is cut into blocks of ``block_length`` SNPs separated by recombination
    hotspots.  Inside a block the population carries a few ancestral haplotypes related by a
    random genealogy (a SNP marks one clade of it, so SNPs on the same clade are in perfect
    LD and nested clades are in partial LD) with unequal frequencies.  Each chromosome copies
    one ancestral haplotype per block, switching rarely inside blocks and freely at hotspots
    (a Li–Stephens-style mosaic).  Two chromosomes make one diploid genotype.

    Returns ``(G, positions)``: an N × J int8 genotype matrix (convert with ``astype(float)``
    before arithmetic) and SNP positions in base pairs.
    """
    n_blocks = int(np.ceil(n_snps / block_length))
    block = np.arange(n_snps) // block_length
    founders = np.zeros((n_founders, n_snps), dtype=np.int8)
    hap_freq = rng.dirichlet(np.ones(n_founders), n_blocks)               # ancestral haplotype frequencies
    for b in range(n_blocks):
        clades, stack = [], [list(rng.permutation(n_founders))]
        while stack:                                                       # random genealogy by splitting
            group = stack.pop()
            if len(group) > 1:
                cut = int(rng.integers(1, len(group)))
                stack += [group[:cut], group[cut:]]
                clades += [group[:cut], group[cut:]]
        for j in np.flatnonzero(block == b):
            founders[clades[rng.integers(len(clades))], j] = 1
    n_hap = 2 * n_samples
    path = np.empty((n_hap, n_snps), dtype=np.int64)
    path[:, 0] = rng.choice(n_founders, n_hap, p=hap_freq[0])
    for j in range(1, n_snps):
        p_switch = 0.9 if block[j] != block[j - 1] else 0.003
        switch = rng.random(n_hap) < p_switch
        path[:, j] = np.where(switch, rng.choice(n_founders, n_hap, p=hap_freq[block[j]]), path[:, j - 1])
    hap = founders[path, np.arange(n_snps)] ^ (rng.random((n_hap, n_snps)) < mutation)
    G = (hap[0::2] + hap[1::2]).astype(np.int8)
    positions = np.cumsum(rng.integers(500, 3000, n_snps))
    return G, positions


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def benjamini_hochberg(pvalues) -> np.ndarray:
    """Benjamini–Hochberg adjusted p-values (q-values): reject where q ≤ α to control the FDR at α."""
    p = np.asarray(pvalues, float)
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(p)
    out[order] = np.minimum(q, 1.0)
    return out


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_genotypes', 'simulate_quantitative_trait', 'simulate_ld_genotypes', 'standardize', 'benjamini_hochberg', 'draw_graph', 'plot_embedding']))

---
## 1 · From Module 06 to biobank GWAS

> 📍 **Where we are.** In [Module 06](06_linear_regression_gwas.ipynb) you scanned 2,000 SNPs one at a time with
> $y=w_0+wx+\epsilon$, computed $t$-statistics and $p$-values, drew a Manhattan plot and met two reasons why a hit is not
> a cause (LD and population structure). Here we scale the same scan up, add **covariates**, and learn the two
> diagnostic plots every GWAS paper shows: the Manhattan plot and the **QQ plot**.

🔬 **Assay box — genotyping a biobank.** *What is measured:* the alleles a person carries at known variable positions.
*How:* a **SNP array** (a glass slide with probes for ~0.5–1 million common variants) measures each person cheaply;
**imputation** then fills in millions of unmeasured variants by matching stretches of the measured genotypes to a
reference panel of sequenced genomes (the haplotype-copying model of Li & Stephens 2003). Whole-exome and whole-genome
sequencing are replacing arrays as costs fall. *Typical sizes:* $N=10^4$–$10^6$ people, $J=10^5$–$10^7$ variants.
*Typical artefacts:* genotype-calling errors (rare variants especially), batch effects between array versions,
poorly imputed variants, cryptic relatives and duplicate samples, and differences in **ancestry** that correlate with
both genotypes and environment.

**Covariates.** A biobank GWAS never regresses the trait on a SNP alone: age, sex, genotyping batch and ancestry
(principal components, Section 2) are always included. With covariate matrix $C$ (plus an intercept), the model for SNP
$j$ is $y = C\gamma + x_j w_j + \epsilon$.

🧮 **Math refresher — adjusting for covariates by "regressing them out" (Frisch–Waugh–Lovell).** Let $P$ project onto
the columns of $C$ (in code: an orthonormal basis $Q$ of $C$ from a QR decomposition, $P = QQ^\top$). Then

$$\hat w_j = \frac{\tilde x_j^\top \tilde y}{\tilde x_j^\top \tilde x_j},\qquad \tilde y = (I-P)\,y,\quad \tilde x_j=(I-P)\,x_j .$$

*In words:* remove from the trait and from the SNP everything the covariates can explain, then do the ordinary
one-SNP regression of Module 06 on the residuals; the slope is *exactly* the multiple-regression coefficient you would
get from the normal equations with $C$ and $x_j$ together. The trick lets us test 5,000 SNPs with two matrix products
instead of 5,000 separate regressions. The residual degrees of freedom shrink by the number of covariates.

In [ ]:
# ── 1.1 A SIMULATED mini-biobank: 1,500 unrelated people × 5,000 SNPs, 25 causal SNPs, h² = 0.3 ──
rng = np.random.default_rng(SEED + 1)
N1, J1 = 1500, 5000
G1, _, freq1 = simulate_genotypes(rng, N1, J1)                 # one population, Hardy–Weinberg proportions
y1, beta1, causal1 = simulate_quantitative_trait(rng, G1, n_causal=25, h2=0.3)
print(f"SIMULATED cohort: {N1} people × {J1} SNPs (genotypes 0/1/2), {len(causal1)} causal SNPs, true h² = 0.3")
print(f"allele frequencies: {freq1.min():.2f}–{freq1.max():.2f};  phenotype mean {y1.mean():+.2f}, variance {y1.var():.2f}")


def assoc_scan(G, y, covariates=None):
    """Per-SNP least-squares association test, adjusted for covariates (Frisch–Waugh–Lovell).

    y and every SNP are residualised on [1, covariates]; the slope of residual-y on residual-SNP is the
    multiple-regression coefficient. Returns (beta, se, chi2, p); SNPs without variation get NaN.
    """
    X = np.asarray(G, float)
    n = len(y)
    C = np.ones((n, 1)) if covariates is None else np.column_stack([np.ones(n), covariates])
    Q, _ = np.linalg.qr(C)                                    # orthonormal basis of the covariate space
    y_r = y - Q @ (Q.T @ y)
    X_r = X - Q @ (Q.T @ X)
    sxx = np.sum(X_r ** 2, axis=0)
    ok = sxx > 1e-8
    beta, se = np.full(X.shape[1], np.nan), np.full(X.shape[1], np.nan)
    beta[ok] = X_r[:, ok].T @ y_r / sxx[ok]
    rss = y_r @ y_r - beta[ok] ** 2 * sxx[ok]                 # residual sum of squares (Module 06, 5.1)
    se[ok] = np.sqrt(rss / (n - C.shape[1] - 1) / sxx[ok])
    chi2 = (beta / se) ** 2                                   # = t²; ≈ chi-square with 1 d.f. under H0 for large N
    return beta, se, chi2, stats.chi2.sf(chi2, df=1)


def lambda_gc(chi2):
    """Genomic-inflation factor: median observed chi² divided by the median of chi²(1 d.f.) = 0.455."""
    return np.nanmedian(chi2) / stats.chi2.ppf(0.5, df=1)


t0 = time.perf_counter()
b1, se1, chi2_1, p1 = assoc_scan(G1, y1)
hits1 = np.flatnonzero(p1 < 0.05 / J1)
print(f"scan of {J1} SNPs in {time.perf_counter() - t0:.2f} s | Bonferroni hits: {len(hits1)} "
      f"({np.isin(hits1, causal1).sum()} truly causal) | lambda_GC = {lambda_gc(chi2_1):.3f}")

🧮 **Math refresher — the QQ plot and $\lambda_{GC}$.** If no SNP had any effect, the $J$ $p$-values would be a sample
from the uniform distribution on [0, 1] (you checked this by shuffling in Module 06). Sort them and plot the observed
$-\log_{10}p_{(i)}$ against the values expected for uniform order statistics, $-\log_{10}\big((i-0.5)/J\big)$: under
the null the points lie on the diagonal. True signals bend the *top right* tip upwards; **systematic inflation** —
confounding, cryptic relatedness, or many tiny true effects — lifts the *whole* curve. The **genomic-inflation factor**
summarises the bulk of the distribution:

$$\lambda_{GC}=\frac{\operatorname{median}(\chi^2_1,\dots,\chi^2_J)}{0.455}.$$

*In words:* $0.455$ is the median of a $\chi^2$ distribution with one degree of freedom (the null distribution of a
squared $z$-score); $\lambda_{GC}\approx1$ means the typical SNP behaves as if it had no effect, $\lambda_{GC}=1.3$ means
the typical test statistic is 30 % too large. Using the median makes it robust to the few true hits.

In [ ]:
# ── 1.2 Manhattan and QQ plots ──
def manhattan_qq(p, causal, title, axes, snps_per_chrom=500):
    """Left: Manhattan plot (pretend chromosomes) with true causal SNPs circled. Right: QQ plot with lambda_GC."""
    J = len(p)
    x = np.arange(J)
    lp = -np.log10(np.clip(p, 1e-300, 1))
    axes[0].scatter(x, lp, s=3, c=np.where((x // snps_per_chrom) % 2 == 0, "tab:blue", "0.6"))
    axes[0].scatter(causal, lp[causal], s=28, facecolors="none", edgecolors="tab:red", lw=1.2,
                    label="true causal SNPs (answer key)")
    axes[0].axhline(-np.log10(0.05 / J), color="k", ls="--", lw=0.8, label="Bonferroni 0.05/J")
    axes[0].set_xlabel("SNP (pretend chromosomes of 500 SNPs)"); axes[0].set_ylabel("−log10 p")
    axes[0].set_title(title); axes[0].legend(fontsize=7, loc="upper right")
    ok = ~np.isnan(p)
    obs = np.sort(lp[ok])[::-1]
    exp = -np.log10((np.arange(1, ok.sum() + 1) - 0.5) / ok.sum())
    axes[1].scatter(exp, obs, s=4, color="tab:blue")
    axes[1].plot([0, exp.max()], [0, exp.max()], "k--", lw=0.8)
    axes[1].set_xlabel("expected −log10 p (uniform)"); axes[1].set_ylabel("observed −log10 p")
    axes[1].set_title(f"QQ plot: λ_GC = {lambda_gc(stats.chi2.isf(np.clip(p[ok], 1e-300, 1), 1)):.2f}")


fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [2.3, 1]})
manhattan_qq(p1, causal1, "SIMULATED homogeneous cohort: 25 causal SNPs, h² = 0.3", axes)
plt.tight_layout(); plt.show()

q1 = benjamini_hochberg(p1)
print(f"Bonferroni (FWER 5 %): {np.sum(p1 < 0.05 / J1)} hits | Benjamini–Hochberg (FDR 5 %): {np.sum(q1 < 0.05)} hits, "
      f"of which {np.isin(np.flatnonzero(q1 < 0.05), causal1).sum()} causal")

🔍 **What just happened?** In a single, homogeneous population the scan behaves as Module 06 promised:
9 SNPs pass Bonferroni and all 9 are truly causal; the QQ plot follows the diagonal and only its top-right tip (the
true signals) lifts off, with $\lambda_{GC}=1.02$. The Benjamini–Hochberg procedure (controlling the *false discovery
rate* instead of the chance of *any* false positive) finds 12 SNPs, all of them causal in this run — the usual
trade-off: more power for a looser guarantee. Sixteen of the 25 causal SNPs are still missed (the circled points below
the line): their effects are too small for 1,500 people.

Two lessons for biobanks: with $N$ in the hundreds of thousands, **standard errors shrink like $1/\sqrt N$**, so
variants explaining 0.01 % of the variance become detectable (GWAS of height now report thousands of loci); and the
threshold $5\times10^{-8}$ is Bonferroni for about one million *independent* tests — LD makes neighbouring tests
correlated, so the effective number of tests is smaller than the number of SNPs.

✏️ **Your turn.** Re-run 1.1–1.2 with `n_causal=500` (same $h^2=0.3$). How many Bonferroni hits remain, and what happens
to $\lambda_{GC}$? (Keep the answer in mind: Section 7 explains why many tiny *true* effects inflate $\lambda_{GC}$.)

---
## 2 · Population structure: spurious hits and principal components

> 📍 **Where we are.** Module 06 (Section 5.5) showed that a mixture of two populations that differ in environment
> produces false hits, and [Module 14](14_pca_dimensionality_reduction.ipynb) (Section 7) that genotype PCs mirror ancestry. Now we put the two together the way
> every GWAS pipeline does: estimate ancestry from the genotypes and include it as covariates.

**The mechanism.** Populations that have been separated for many generations differ slightly in allele frequency at
almost every SNP (genetic drift). If they also differ in an environmental factor that affects the trait — diet,
climate, socioeconomic conditions, or even the recruitment centre — then *every* SNP whose frequency differs between
populations correlates with the trait. The result is genome-wide inflation and false "hits" that have nothing to do with
biology.

🧮 **Math refresher — $F_{ST}$ and the Balding–Nichols model.** $F_{ST}$ is the fraction of genetic variance that lies
*between* populations. `simulate_genotypes` draws each population's allele frequency at a SNP from a Beta distribution
centred on the ancestral frequency $p$ with variance $F_{ST}\,p(1-p)$. Human continental groups differ by
$F_{ST}\approx0.05$–$0.15$, populations within a continent by $F_{ST}\approx0.001$–$0.01$; we use $0.05$.

In [ ]:
# ── 2.1 Three SIMULATED populations (F_ST = 0.05) that also differ in an environmental factor ──
rng = np.random.default_rng(SEED + 2)
N2, J2 = 1500, 5000
G2, pop2, freq2 = simulate_genotypes(rng, N2, J2, n_pops=3, fst=0.05)
y2_genetic, beta2, causal2 = simulate_quantitative_trait(rng, G2, n_causal=20, h2=0.3)
environment = np.array([0.0, 0.5, -0.5])[pop2]           # e.g. diet: population 1 +0.5 SD, population 2 −0.5 SD
y2 = y2_genetic + environment
print("people per population:", np.bincount(pop2), "| mean phenotype per population:",
      np.round([y2[pop2 == k].mean() for k in range(3)], 2))
print("allele-frequency difference between populations 0 and 1: median |Δp| =",
      round(float(np.median(np.abs(freq2[0] - freq2[1]))), 3))

b2_naive, _, chi2_2naive, p2_naive = assoc_scan(G2, y2)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [2.3, 1]})
manhattan_qq(p2_naive, causal2, "SIMULATED structured cohort, naive scan (no ancestry covariates)", axes)
plt.tight_layout(); plt.show()
hits = np.flatnonzero(p2_naive < 0.05 / J2)
print(f"naive scan: {len(hits)} Bonferroni hits, {np.isin(hits, causal2).sum()} causal, "
      f"{(~np.isin(hits, causal2)).sum()} FALSE; lambda_GC = {lambda_gc(chi2_2naive):.2f}")

🔍 **What just happened?** The Manhattan plot looks like a city skyline *everywhere*: 384 SNPs pass Bonferroni, but
only 7 of them are causal — 377 are false positives — and the whole QQ curve leaves the diagonal from the start
($\lambda_{GC}=7.03$). Nothing about the biology changed relative to Section 1; we only mixed three populations that
differ in allele frequencies *and* environment.

**The fix: estimate ancestry from the genotypes.** Standardise every SNP ($z_{ij}=(g_{ij}-2p_j)/\sqrt{2p_j(1-p_j)}$) and
form the $N\times N$ **genetic relationship matrix** (GRM)

$$K=\frac{1}{J}\,ZZ^\top,\qquad K_{ij}=\frac1J\sum_{k=1}^{J} z_{ik}z_{jk}.$$

*In words:* $K_{ij}$ is the average, over SNPs, of how similar persons $i$ and $j$ are at each SNP (positive when they
share alleles that are rare in the sample). Its top eigenvectors are exactly the principal-component scores of
Module 14, computed through the $N\times N$ matrix instead of the $J\times J$ covariance — the "SVD trick" for
$J\gg N$. We will reuse the same eigendecomposition for the mixed model in Section 3.

In [ ]:
# ── 2.2 The GRM, its eigendecomposition and the genotype PCs ──
def standardise_genotypes(G):
    """Centre each SNP at 2p and scale to unit variance; drop SNPs without variation."""
    X = np.asarray(G, float)
    m, s = X.mean(axis=0), X.std(axis=0)
    ok = s > 0
    return (X[:, ok] - m[ok]) / s[ok]


t0 = time.perf_counter()
Z2 = standardise_genotypes(G2)
K2 = Z2 @ Z2.T / Z2.shape[1]                       # N × N genetic relationship matrix
s2, U2 = np.linalg.eigh(K2)                        # ascending order (Module 14!) → reverse
s2, U2 = np.clip(s2[::-1], 0, None), U2[:, ::-1]
pcs2 = U2[:, :10] * np.sqrt(s2[:10])               # PC scores
print(f"GRM {K2.shape} and its eigendecomposition in {time.perf_counter() - t0:.1f} s")
print("largest eigenvalues of K:", np.round(s2[:5], 2), " (sum of all = trace = N =", round(float(s2.sum())), ")")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.3))
plot_embedding(pcs2[:, :2], np.array([f"population {k}" for k in pop2]), ax=axes[0], s=5,
               title="PC1 vs PC2 of the SIMULATED genotypes")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[1].bar(np.arange(1, 21), s2[:20], color="tab:blue")
axes[1].set_xticks([1, 5, 10, 15, 20])
axes[1].set_xlabel("eigenvalue rank"); axes[1].set_ylabel("eigenvalue of K")
axes[1].set_title("two large eigenvalues = three populations")
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.3 🧪 Ground-truth check: false positives and lambda_GC with 0, 1, 2, 5, 10 PCs as covariates ──
rows = []
for k in [0, 1, 2, 5, 10]:
    _, _, chi2_k, p_k = assoc_scan(G2, y2, pcs2[:, :k] if k else None)
    hits = np.flatnonzero(p_k < 0.05 / J2)
    rows.append({"PCs": k, "lambda_GC": round(lambda_gc(chi2_k), 3), "Bonferroni hits": len(hits),
                 "true positives": int(np.isin(hits, causal2).sum()),
                 "false positives": int((~np.isin(hits, causal2)).sum())})
    if k == 2:
        p2_pc = p_k
pd.DataFrame(rows).set_index("PCs")

🔍 **What just happened?** The first two PCs separate the three populations perfectly: two eigenvalues (47.4 and 46.4)
stand far above a flat bulk near 2.2 — the level that random genotypes alone produce (for $N/J=0.3$, random-matrix theory
puts the upper edge of the noise bulk at $(1+\sqrt{0.3})^2\approx2.4$). Adding the two PCs as covariates removes almost
all of the inflation: $\lambda_{GC}$ falls from 7.03 to 1.01 and the false positives from 377 to 1, while the true
positives *increase* (7 → 12) because the population differences that the PCs absorb no longer swamp the real signals.
One PC is not enough ($\lambda_{GC}=1.55$): three populations need two axes. More PCs change little here
because there are only three populations; real analyses — following the EIGENSTRAT approach of Price *et al.*
(2006) — typically include 10–20 PCs plus recruitment centre.

🧪 **Ground truth.** We know which 20 SNPs are causal, so "false positive" is not a guess. In a real study the only
evidence would be the QQ plot and $\lambda_{GC}$.

⚠️ **PCs capture broad structure, not everything.** Fine-scale structure (neighbouring villages), **cryptic
relatedness** (cousins in the cohort) and family-shared environments are not well described by a handful of PCs.
Mixed models (Section 3) handle all of these with the full GRM — and family-based designs (comparing siblings) remove
confounding by *any* between-family factor.

✏️ **Your turn.** Set the environment vector to `[0, 0, 0]` in 2.1 and rerun 2.1 and 2.3. Is the naive scan still
inflated? (Hint: the causal SNPs also differ in frequency between populations, so the *genetic* mean differs too —
real "stratification" can be partly genetic.)

---
## 3 · Linear mixed models: heritability and association

> 📍 **Where we are.** PCs remove the big axes of ancestry. A **linear mixed model** (LMM) goes further: it lets
> *every* pair of people be similar according to their genome-wide relatedness $K_{ij}$. The same model answers a
> second question for free — **how much of the trait variance is explained by all SNPs together** (SNP heritability).

**The model.** For $N$ people with fixed-effect covariates $X$ (intercept, PCs, age, …):

$$y = X\beta + g + \epsilon,\qquad g\sim\mathcal N(0,\ \sigma_g^2K),\qquad \epsilon\sim\mathcal N(0,\ \sigma_e^2I)
\quad\Longrightarrow\quad y\sim\mathcal N\big(X\beta,\ \sigma_g^2K+\sigma_e^2I\big).$$

*In words:* each person's trait is a covariate part, a **random genetic value** $g_i$ and noise. Genetic values of two
people are correlated in proportion to how much of their genome they share ($K_{ij}$); because the GRM is built from
standardised genotypes its diagonal averages 1, so $\sigma_g^2$ is the genetic variance and the **SNP heritability** is
$h^2=\sigma_g^2/(\sigma_g^2+\sigma_e^2)$. Only two variance parameters are estimated, no matter how many SNPs there are.

> 🔗 **Builds on — [Module 07](07_ridge_lasso_sparse_regression.ipynb) (ridge = MAP with a Gaussian prior).** Write $g = Zu/\sqrt J$ with one effect $u_k$ per
> standardised SNP and the prior $u_k\sim\mathcal N(0,\sigma_g^2)$. Then $\operatorname{Cov}(g)=\sigma_g^2ZZ^\top/J=\sigma_g^2K$ —
> the LMM *is* a Bayesian linear regression on all SNPs at once, and the posterior mean of $u$ (the **BLUP**, best linear
> unbiased predictor) is ridge regression with $\lambda=J\sigma_e^2/\sigma_g^2=J(1-h^2)/h^2$. What the LMM adds is that it
> *estimates the prior variance from the data* by maximum likelihood — **empirical Bayes**, the type-II maximum
> likelihood of [Module 21](21_bayesian_learning_mcmc.ipynb). We will use exactly this λ for polygenic scores in Section 5.

🧮 **Math refresher — what the entries of $K$ mean.** For unrelated people $K_{ij}$ scatters around 0 with standard
deviation $\approx1/\sqrt{J}$ (with independent SNPs); for full siblings or parent–offspring it is about 0.5 (twice the
kinship coefficient), for first cousins about 0.125, for a person with themselves $\approx1+F$ ($F$ = inbreeding).

In [ ]:
# ── 3.1 The GRM of the homogeneous cohort of Section 1, and a new POLYGENIC trait (500 causal SNPs, h² = 0.5) ──
t0 = time.perf_counter()
Z1 = standardise_genotypes(G1)
K1 = Z1 @ Z1.T / Z1.shape[1]
s1, U1 = np.linalg.eigh(K1)
s1, U1 = np.clip(s1[::-1], 0, None), U1[:, ::-1]
print(f"GRM and eigendecomposition: {time.perf_counter() - t0:.1f} s")
off = K1[np.triu_indices(N1, 1)]
print(f"diagonal mean {np.diag(K1).mean():.3f} | off-diagonal mean {off.mean():+.5f}, sd {off.std():.4f} "
      f"(1/sqrt(J) = {1 / np.sqrt(Z1.shape[1]):.4f}) | largest {off.max():.3f}")

rng = np.random.default_rng(SEED + 3)
G1f = G1.astype(float)                                         # float copy: faster repeated trait simulation
y1p, beta1p, causal1p = simulate_quantitative_trait(rng, G1f, n_causal=500, h2=0.5)
print("SIMULATED polygenic trait: 500 causal SNPs, true h² = 0.5")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.0))
axes[0].hist(off, bins=80, color="0.6")
axes[0].set_xlabel("$K_{ij}$ (i ≠ j)"); axes[0].set_ylabel("number of pairs")
axes[0].set_title("unrelated people: K_ij ≈ 0 ± 1/√J")
axes[1].semilogy(np.arange(1, 51), s1[:50], "o-", ms=3, label="homogeneous cohort (Section 1)")
axes[1].semilogy(np.arange(1, 51), s2[:50], "s-", ms=3, label="three populations (Section 2)")
axes[1].set_xlabel("eigenvalue rank"); axes[1].set_ylabel("eigenvalue of K (log)")
axes[1].legend(fontsize=7); axes[1].set_title("eigenvalues of the two GRMs")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The diagonal of $K$ averages 1.000 and the off-diagonal entries scatter around zero with
standard deviation 0.0141, exactly $1/\sqrt J$ (largest value 0.069) — nobody in this cohort is related. The eigenvalue spectrum of the
homogeneous cohort is flat, while the structured cohort has two eigenvalues that tower above the rest.

**Maximum likelihood via an eigen-rotation (the EMMA / FaST-LMM idea).** Writing $\delta=\sigma_e^2/\sigma_g^2$, the covariance is
$V=\sigma_g^2(K+\delta I)$. Evaluating the Gaussian likelihood naively needs $V^{-1}$ and $\log|V|$ — an $O(N^3)$
operation for *every* trial value of $\delta$. EMMA (Kang *et al.* 2008) used a spectral decomposition so that the
likelihood can be evaluated cheaply for every $\delta$; FaST-LMM (Lippert *et al.* 2011) re-used a single rotation for
all SNPs of a scan and showed that a GRM built from fewer SNPs than people (low rank) makes the cost linear in $N$.
The core fact is that one eigendecomposition $K=USU^\top$ diagonalises $V$ for all $\delta$ at once:

$$U^\top y\ \sim\ \mathcal N\!\big(U^\top X\beta,\ \sigma_g^2\,(S+\delta I)\big)\quad\text{— a diagonal covariance.}$$

*In words:* after rotating the people into the eigenbasis of $K$, the $N$ rotated observations are **independent**,
the $i$-th with variance $\sigma_g^2(s_i+\delta)$. For fixed $\delta$, $\beta$ is a *weighted* least-squares fit with
weights $1/(s_i+\delta)$, and $\sigma_g^2$ has a closed form, so the log-likelihood becomes a function of one number:

$$\ell(\delta)=-\tfrac12\Big[\,n\log\!\big(2\pi\hat\sigma_g^2\big)+\textstyle\sum_i\log(s_i+\delta)+n\Big],\qquad
\hat\sigma_g^2=\frac1n\sum_i\frac{(\tilde y_i-\tilde x_i^\top\hat\beta)^2}{s_i+\delta}.$$

*In words:* the likelihood rewards a $\delta$ that makes the weighted residuals small without paying too much in the
log-determinant term $\sum_i\log(s_i+\delta)$. We maximise over a single variable with a 1-D search. **REML** (restricted
maximum likelihood, the default of GCTA) replaces $n$ by $n-p$ ($p$ = number of fixed effects) and subtracts
$\tfrac12\log|X^\top H^{-1}X|$ with $H=K+\delta I$; this removes the small downward bias of ML caused by estimating
$\beta$ from the same data (just as dividing by $n-1$ instead of $n$ unbiases a sample variance).

In [ ]:
# ── 3.2 The profile (RE)ML log-likelihood in rotated coordinates ──
def lmm_loglik(h2, Uy, UX, s, reml=True):
    """Profile log-likelihood of y ~ N(X b, sg2 (K + delta I)) at heritability h2 (delta = (1 - h2) / h2).

    Uy = U'y and UX = U'X are the data rotated by the eigenvectors U of K; s are its eigenvalues.
    Returns (log-likelihood, beta_hat, sigma_g^2_hat).
    """
    delta = (1 - h2) / h2
    w = 1.0 / (s + delta)                                   # inverse variances of the rotated observations
    XtWX = UX.T @ (w[:, None] * UX)
    beta = np.linalg.solve(XtWX, UX.T @ (w * Uy))           # weighted least squares
    r = Uy - UX @ beta
    n, p = UX.shape
    df = n - p if reml else n
    sg2 = np.sum(w * r ** 2) / df
    ll = -0.5 * (df * np.log(2 * np.pi * sg2) + np.sum(np.log(s + delta)) + df)
    if reml:
        ll -= 0.5 * np.linalg.slogdet(XtWX)[1]
    return ll, beta, sg2


def fit_lmm(Uy, UX, s, reml=True):
    """Maximise the profile likelihood over h2 in (0, 1) with a bounded 1-D search; returns (h2_hat, max loglik)."""
    res = optimize.minimize_scalar(lambda h: -lmm_loglik(h, Uy, UX, s, reml)[0],
                                   bounds=(1e-3, 1 - 1e-3), method="bounded")
    return res.x, -res.fun

In [ ]:
# ── 3.3 Fit h² of the polygenic trait: estimate, likelihood-ratio interval, standard error ──
Uy1, UX1 = U1.T @ y1p, U1.T @ np.ones((N1, 1))              # rotate once; every likelihood evaluation is then O(N)
grid = np.linspace(0.02, 0.98, 97)
ll_grid = np.array([lmm_loglik(h, Uy1, UX1, s1)[0] for h in grid])
h2_hat, ll_max = fit_lmm(Uy1, UX1, s1)
h2_ml, _ = fit_lmm(Uy1, UX1, s1, reml=False)
ci = grid[ll_grid >= ll_max - stats.chi2.ppf(0.95, 1) / 2]  # likelihood-ratio 95 % confidence interval
eps = 1e-3                                                   # curvature of the log-likelihood → standard error
curv = (lmm_loglik(h2_hat + eps, Uy1, UX1, s1)[0] - 2 * ll_max + lmm_loglik(h2_hat - eps, Uy1, UX1, s1)[0]) / eps ** 2
se_h2 = 1 / np.sqrt(-curv)
print(f"REML h² = {h2_hat:.3f} ± {se_h2:.3f} (SE from curvature) | 95 % LR interval [{ci.min():.2f}, {ci.max():.2f}] "
      f"| ML h² = {h2_ml:.3f} | TRUE h² = 0.5")
print(f"rule of thumb SE ≈ sqrt(2) / (N · sd(K_ij)) = {np.sqrt(2) / (N1 * off.std()):.3f}")

fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.plot(grid, ll_grid - ll_max, color="tab:blue")
ax.axhline(-stats.chi2.ppf(0.95, 1) / 2, color="0.5", ls=":", label="95 % interval cut-off (−1.92)")
ax.axvline(0.5, color="tab:red", ls="--", label="true h² = 0.5")
ax.axvline(h2_hat, color="tab:blue", ls="-.", label=f"REML estimate {h2_hat:.2f}")
ax.set_ylim(-12, 0.5); ax.set_xlabel("h²"); ax.set_ylabel("log-likelihood − maximum")
ax.set_title("profile REML likelihood of SNP heritability (SIMULATED)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** From 1,500 unrelated people and nothing but their genotypes and one trait, the mixed model
estimates $\hat h^2=0.48$ with a 95 % interval of [0.35, 0.60] that contains the true value 0.5. (ML gives 0.480,
practically the same: with a single fixed effect, the intercept, the REML correction is negligible; it matters when many
covariates are fitted.) The standard error from the curvature of the log-likelihood (0.067) agrees with a simple rule of
thumb, $\mathrm{SE}\approx\sqrt2/(N\cdot\mathrm{sd}(K_{ij}))=0.067$ (Visscher *et al.* 2014), which follows from
regressing products of phenotypes $y_iy_j$ on $K_{ij}$ (Haseman–Elston regression; Haseman & Elston 1972): the more the relatedness varies between pairs
and the more pairs there are, the better $h^2$ is pinned down. For unrelated people $\mathrm{sd}(K_{ij})$ is tiny, which
is why SNP heritability needs thousands of samples, while twin studies (where $K_{ij}$ is 0.5 or 1) need far fewer.

A single interval can be lucky. Because we know the truth, we can check the *procedure*: simulate many traits on the same
genotypes and see how often the interval covers 0.5.

In [ ]:
# ── 3.4 🧪 Ground-truth check: 50 replicate traits on the same genotypes ──
t0 = time.perf_counter()
est, covered = [], []
for _ in range(50):
    y_rep, _, _ = simulate_quantitative_trait(rng, G1f, n_causal=500, h2=0.5)
    Uy = U1.T @ y_rep
    h_rep, ll_rep = fit_lmm(Uy, UX1, s1)
    est.append(h_rep)
    covered.append(2 * (ll_rep - lmm_loglik(0.5, Uy, UX1, s1)[0]) <= stats.chi2.ppf(0.95, 1))
est = np.array(est)
print(f"50 replicates in {time.perf_counter() - t0:.1f} s: mean h² = {est.mean():.3f} (truth 0.5), "
      f"SD = {est.std(ddof=1):.3f} (curvature SE above: {se_h2:.3f}), 95 % intervals cover the truth in "
      f"{100 * np.mean(covered):.0f} % of replicates")

hits_p = np.flatnonzero(assoc_scan(G1f, y1p)[3] < 0.05 / J1)
Xh = np.column_stack([np.ones(N1), G1f[:, hits_p]])
r2_hits = 1 - np.var(y1p - Xh @ np.linalg.lstsq(Xh, y1p, rcond=None)[0]) / np.var(y1p)
print(f"'missing heritability': {len(hits_p)} SNPs pass Bonferroni for this trait and together explain "
      f"R² = {r2_hits:.3f} — versus h² = {h2_hat:.2f} captured by ALL SNPs in the mixed model")

🔍 **What just happened?** Over 50 replicate studies the REML estimate averages 0.486 (truth 0.5) with a spread of
0.056 — close to the curvature-based SE of 0.067 — and the 95 % likelihood intervals cover the truth in 96 % of
replicates: the
estimator is (nearly) unbiased and its uncertainty is honestly reported.

The last line is the **missing-heritability** story of Yang *et al.* (2010) in miniature. With 500 causal SNPs each
explaining about 0.1 % of the variance, **not a single SNP** passes Bonferroni (so GWAS hits explain nothing), while
the mixed model, which never asks *which* SNPs matter, recovers about half of the variance. The heritability is not missing; it is
spread over many variants that are individually too weak to detect at this sample size.

⚠️ **What SNP heritability is and is not.** $h^2_{SNP}$ is the fraction of variance explained by *additive* effects of
the genotyped SNPs (and variants in LD with them) *in this population and environment*. It is usually lower than twin
heritability (rare variants and imperfect tagging are missed), it can be inflated by uncorrected stratification or
shared environment among relatives, and it says nothing about how changeable a trait is by environment or
treatment — highly heritable traits can be very responsive to intervention.

**Mixed-model association testing.** The same machinery gives an association test that corrects for structure *and*
relatedness: estimate $\delta$ once under the null model (no SNP), rotate the phenotype and every SNP by $U^\top$,
scale each rotated observation by $1/\sqrt{s_i+\hat\delta}$ — after which the noise is independent with equal variance —
and run the ordinary scan of Section 1 on the transformed data. Re-using $\hat\delta$ for all SNPs (instead of
re-estimating it for each SNP) is the approximation of EMMAX (Kang *et al.* 2010); it is accurate because a single SNP
explains very little variance.

In [ ]:
# ── 3.5 🧪 LMM association on the structured cohort of Section 2 (re-using its eigendecomposition) ──
def lmm_scan(UG, Uy, UX, s, delta):
    """EMMAX-style scan: whiten the rotated data with 1/sqrt(s + delta), then run assoc_scan's algebra."""
    w = 1.0 / np.sqrt(s + delta)
    ys, Gs, Cs = w * Uy, w[:, None] * UG, w[:, None] * UX
    Q, _ = np.linalg.qr(Cs)                                  # covariates (here: the whitened intercept)
    y_r = ys - Q @ (Q.T @ ys)
    G_r = Gs - Q @ (Q.T @ Gs)
    sxx = np.sum(G_r ** 2, axis=0)
    b = G_r.T @ y_r / sxx
    rss = y_r @ y_r - b ** 2 * sxx
    chi2 = b ** 2 * sxx / (rss / (len(ys) - UX.shape[1] - 1))
    return chi2, stats.chi2.sf(chi2, 1)


UX2, Uy2 = U2.T @ np.ones((N2, 1)), U2.T @ y2
h2_null, _ = fit_lmm(Uy2, UX2, s2)
UG2 = U2.T @ G2.astype(float)
chi2_lmm, p_lmm = lmm_scan(UG2, Uy2, UX2, s2, (1 - h2_null) / h2_null)
rows = []
for name, chi2_, p_ in [("naive", chi2_2naive, p2_naive), ("2 PCs", stats.chi2.isf(p2_pc, 1), p2_pc),
                        ("LMM (full GRM)", chi2_lmm, p_lmm)]:
    hits = np.flatnonzero(p_ < 0.05 / J2)
    rows.append({"method": name, "lambda_GC": round(lambda_gc(chi2_), 3), "true positives": int(np.isin(hits, causal2).sum()),
                 "false positives": int((~np.isin(hits, causal2)).sum()),
                 "median chi² at causal SNPs": round(float(np.median(chi2_[causal2])), 1)})
print(f"null-model h² (structured cohort) = {h2_null:.2f}")
pd.DataFrame(rows).set_index("method")

🔍 **What just happened?** Without being told the population labels or given any PCs, the mixed model removes the
confounding: $\lambda_{GC}$ drops from 7.03 to 0.93, with 11 true and 0 false positives (2 PCs: 12 true, 1 false), and
the median $\chi^2$ at the causal SNPs is as high as with PCs (22.1 vs 22.7). The GRM's two large eigenvalues
*are* the population axes, so the random effect $g$ soaks up the population mean differences together with the
polygenic background.

Notice that $\lambda_{GC}$ is slightly **below 1**. Each tested SNP is also part of the GRM, so the random effect
partly "explains away" the SNP's own signal (*proximal contamination*; Listgarten *et al.* 2012 and Yang *et al.* 2014
discuss it). Production tools — BOLT-LMM (Loh *et al.* 2015), REGENIE, fastGWA — therefore build the polygenic
background from all chromosomes **except** the one being tested ("leave one chromosome out", LOCO), which also
*increases* power for polygenic traits because the background prediction removes noise from $y$.

✏️ **Your turn (LOCO).** Split the 5,000 SNPs of the structured cohort into two halves (pretend chromosomes A and B).
Build the GRM from half B only, fit the null model, and test the SNPs of half A — then swap. Is $\lambda_{GC}$ closer
to 1 than with the full GRM? (Two extra eigendecompositions of a 1,500 × 1,500 matrix take a few seconds each.)

### 📄 Research connection — Mixed models and heritability

- **What the field does.** Mixed models came to human genetics from animal breeding (Henderson's BLUP). Yang *et al.* (2010) estimated the
variance explained by all common SNPs in unrelated people (about 45 % for height), launching **SNP heritability**
and GCTA (Yang *et al.* 2011). For association, EMMA (Kang *et al.* 2008) evaluated the likelihood through a
spectral decomposition, EMMAX (Kang *et al.* 2010) re-used the null-model variance ratio, FaST-LMM (Lippert *et al.*
2011) re-used one rotation for all SNPs and exploited low-rank GRMs, and BOLT-LMM (Loh *et al.* 2015) scaled
mixed models to biobanks with iterative solvers, LOCO and a mixture prior on SNP effects. SAIGE handles binary
traits with case–control imbalance; REGENIE replaces the GRM by ridge regression on blocks of SNPs.
- **What this notebook builds.** A complete EMMA/FaST-LMM-style REML estimator (profile likelihood over one parameter after rotating by the
eigenvectors of $K$), likelihood-ratio intervals checked for coverage over 50 replicates, and an EMMAX-style
association scan compared with PC correction. At biobank scale an $O(N^3)$ eigendecomposition is impossible
($N=500{,}000$), which is why BOLT-LMM, fastGWA and REGENIE avoid it; the statistical model is the same.
- **Where the frontier is.** Heritability is partitioned by annotation, allele frequency and LD (stratified LDSC, LDAK, GREML-LDMS), with
active debate about how effect sizes depend on frequency and LD. Within-family GWAS separate direct genetic effects
from indirect ones (parental "genetic nurture"), assortative mating and stratification — estimates for behavioural
traits shrink markedly within families. Biobank whole-genome sequencing brings rare variants into these models.
- **Read next.** Yang *et al.* 2010, *Common SNPs explain a large proportion of the heritability for human height*;
Lippert *et al.* 2011, *FaST linear mixed models for genome-wide association studies*;
Loh *et al.* 2015, *Efficient Bayesian mixed-model analysis increases association power in large cohorts*.

In [ ]:
# ── 3.6 Free memory: the large N × N and N × J matrices of Sections 2–3 are no longer needed ──
del Z1, K1, Z2, K2, UG2, U1, G1f
print("freed the GRMs and rotated genotype matrices")

---
## 4 · Linkage disequilibrium and statistical fine-mapping

> 📍 **Where we are.** A GWAS hit is a *region*, not a variant. Module 06 explained why (neighbouring SNPs are inherited
> together) but used independent SNPs. Now we simulate realistic **linkage disequilibrium** (LD) and ask the
> fine-mapping question: *which* SNPs in the region are causal, and how sure can we be?

**Where LD comes from.** Recombination shuffles chromosomes only at a few places per generation, concentrated in
**hotspots**. Between hotspots, stretches of DNA are inherited as blocks, so the population carries a limited number
of ancestral **haplotypes** per block. `simulate_ld_genotypes` builds exactly this: each chromosome is a mosaic that
copies one of 8 ancestral haplotypes per 40-SNP block (the copying model of Li & Stephens 2003), switching freely at
hotspots. LD between SNPs $j$ and $k$ is measured by $r^2_{jk}$, the squared correlation of their genotypes.

In [ ]:
# ── 4.1 SIMULATED LD: 2,000 people × 400 SNPs in 10 blocks of 40 ──
rng = np.random.default_rng(SEED + 4)
G_ld, pos_ld = simulate_ld_genotypes(rng, n_samples=2000, n_snps=400, block_length=40)
X_ld = G_ld.astype(float)
maf_ld = np.minimum(X_ld.mean(0) / 2, 1 - X_ld.mean(0) / 2)
R_ld = np.corrcoef(X_ld.T)
block = np.arange(400) // 40
same = (block[:, None] == block[None, :]) & ~np.eye(400, dtype=bool)
print(f"MAF range {maf_ld.min():.3f}–{maf_ld.max():.3f} | mean r² within blocks {np.mean(R_ld[same] ** 2):.3f}, "
      f"between neighbouring blocks {np.mean(R_ld[np.abs(block[:, None] - block[None, :]) == 1] ** 2):.4f}")
print(f"SNP pairs with r² > 0.8 within blocks: {np.mean(R_ld[same] ** 2 > 0.8):.1%}")

fig, ax = plt.subplots(figsize=(5.2, 4.2))
im = ax.imshow(R_ld[100:220, 100:220] ** 2, cmap="Reds", vmin=0, vmax=1)
for b in (120, 160, 200):
    ax.axhline(b - 100 - 0.5, color="tab:blue", lw=0.6); ax.axvline(b - 100 - 0.5, color="tab:blue", lw=0.6)
ax.set_xticks([0, 40, 80, 119]); ax.set_xticklabels([100, 140, 180, 219])
ax.set_yticks([0, 40, 80, 119]); ax.set_yticklabels([100, 140, 180, 219])
ax.set_xlabel("SNP"); ax.set_ylabel("SNP")
ax.set_title("LD (r²) of SNPs 100–219: blocks between hotspots (SIMULATED)")
plt.colorbar(im, ax=ax, shrink=0.8, label="r²")
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.2 One causal SNP → a tower of associated SNPs ──
region = np.arange(120, 200)                                   # two LD blocks = our "fine-mapping region"
Xr = (X_ld[:, region] - X_ld[:, region].mean(0)) / X_ld[:, region].std(0)
n_ld = len(Xr)
n_proxies = (R_ld[np.ix_(region, region)] ** 2 > 0.5).sum(axis=0)
j_causal = 40 + int(np.argmax(n_proxies[40:]))                 # a SNP of the second block with many LD partners
y_ld = np.sqrt(0.03) * Xr[:, j_causal] + rng.normal(0, np.sqrt(0.97), n_ld)   # the SNP explains 3 % of variance
y_ld = (y_ld - y_ld.mean()) / y_ld.std()
b_reg, se_reg, chi2_reg, p_reg = assoc_scan(Xr, y_ld)
lead = int(np.nanargmax(chi2_reg))
r2_with_causal = R_ld[region, region[j_causal]] ** 2
print(f"causal SNP = region index {j_causal} (MAF {maf_ld[region[j_causal]]:.2f}); lead SNP = {lead} "
      f"(r² with causal {r2_with_causal[lead]:.2f}); {np.sum(p_reg < 5e-8)} SNPs reach p < 5e-8")

fig, ax = plt.subplots(figsize=(8.5, 3.1))
sc = ax.scatter(np.arange(len(region)), -np.log10(p_reg), c=r2_with_causal, cmap="viridis", s=22, vmin=0, vmax=1)
ax.scatter(j_causal, -np.log10(p_reg[j_causal]), s=150, facecolors="none", edgecolors="tab:red", lw=1.5,
           label="causal SNP (answer key)")
ax.axhline(-np.log10(5e-8), color="k", ls="--", lw=0.8, label="5e-8")
ax.axvline(39.5, color="0.7", lw=0.8)
ax.set_xlabel("SNP in region (two LD blocks; grey line = recombination hotspot)"); ax.set_ylabel("−log10 p")
ax.set_title("regional association plot, coloured by r² with the causal SNP (SIMULATED)")
plt.colorbar(sc, ax=ax, label="r² with causal"); ax.legend(fontsize=7, loc="center left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** LD in our simulation is strong inside blocks (mean $r^2=0.18$, with 7.8 % of pairs above
0.8) and essentially absent between blocks (0.002) — the checkerboard of the heat map. A single causal SNP explaining 3 %
of the variance produces not one point but a **tower**: 6 SNPs reach $p<5\times10^{-8}$, their heights tracking their
$r^2$ with the causal SNP, and SNPs with weaker LD form the lower floors. Every SNP in the other block stays flat. Here
the lead SNP happens to be the causal one; with a slightly different noise draw one of its proxies ($r^2\ge0.9$)
could just as well have won.

**Fine-mapping as Bayesian model comparison.** Assume (for now) that exactly one SNP in the region is causal. For each
SNP $j$, compare the model "SNP $j$ has effect $\beta\sim\mathcal N(0,W)$" with "no effect". If $\hat\beta_j$ has sampling
variance $V_j$ (its squared standard error), then under "no effect" $\hat\beta_j\sim\mathcal N(0,V_j)$ and under "effect"
$\hat\beta_j\sim\mathcal N(0,V_j+W)$. The ratio of the two densities is the **approximate Bayes factor** of Wakefield
(2009):

$$\mathrm{ABF}_j=\frac{\mathcal N(\hat\beta_j;0,V_j+W)}{\mathcal N(\hat\beta_j;0,V_j)}
=\sqrt{\frac{V_j}{V_j+W}}\;\exp\!\Big(\frac{z_j^2}{2}\,\frac{W}{V_j+W}\Big),\qquad z_j=\hat\beta_j/\sqrt{V_j}.$$

*In words:* the evidence for SNP $j$ grows exponentially with its squared $z$-score; the square-root factor is an
Occam penalty for the extra parameter. With a uniform prior over which SNP is causal, Bayes' rule ([Module 02](02_probability_for_biologists.ipynb)) gives
the **posterior inclusion probability** $\mathrm{PIP}_j=\mathrm{ABF}_j/\sum_k\mathrm{ABF}_k$. A **95 % credible set**
is the smallest set of SNPs whose PIPs add up to at least 0.95: "the causal SNP is in this set with 95 % posterior
probability" (Maller *et al.* 2012).

> 🔗 **Builds on — [Module 21](21_bayesian_learning_mcmc.ipynb) (Bayesian inference).** This is the same Bayesian logic
> as there: prior × likelihood, normalised over the hypotheses. Here the hypothesis space ("which SNP is causal") is
> small enough to enumerate, so no MCMC is needed.

In [ ]:
# ── 4.3 Single-effect regression: Bayes factors, PIPs and a 95 % credible set ──
def single_effect_regression(X, y, sigma2=1.0, prior_var=0.15 ** 2):
    """One causal SNP among the columns of X (standardised). Returns (pip, post_mean, post_var, log_bf).

    Uses the Gaussian-prior Bayes factor of Wakefield (2009) with residual variance sigma2 and prior variance
    prior_var on the per-SD effect; post_mean / post_var are the posterior of the effect if SNP j is the causal one.
    """
    xtx = np.sum(X ** 2, axis=0)
    bhat = X.T @ y / xtx
    V = sigma2 / xtx                                        # sampling variance of each marginal estimate
    log_bf = 0.5 * np.log(V / (V + prior_var)) + 0.5 * (bhat ** 2 / V) * prior_var / (V + prior_var)
    pip = np.exp(log_bf - log_bf.max())
    pip /= pip.sum()                                        # Bayes' rule with a uniform prior over SNPs
    post_var = 1 / (1 / V + 1 / prior_var)
    return pip, post_var * bhat / V, post_var, log_bf


def credible_set(pip, level=0.95):
    """Smallest set of SNPs whose PIPs sum to at least `level` (indices, most probable first)."""
    order = np.argsort(-pip)
    return order[:np.searchsorted(np.cumsum(pip[order]), level) + 1]


pip, _, _, _ = single_effect_regression(Xr, y_ld)
cs = credible_set(pip)
print(f"95 % credible set: {len(cs)} SNPs {sorted(cs.tolist())}; contains the causal SNP {j_causal}: {j_causal in cs}")
print("PIPs of the set:", np.round(pip[cs], 3), "| min r² between the set and the causal SNP:",
      round(float(r2_with_causal[cs].min()), 2))

In [ ]:
# ── 4.4 🧪 Ground-truth check: credible-set coverage and size over 200 simulated causal SNPs ──
covered, sizes = [], []
for _ in range(200):
    j = rng.integers(len(region))
    y_sim = np.sqrt(0.02) * Xr[:, j] + rng.normal(0, np.sqrt(0.98), n_ld)
    y_sim = (y_sim - y_sim.mean()) / y_sim.std()
    cs_sim = credible_set(single_effect_regression(Xr, y_sim)[0])
    covered.append(j in cs_sim); sizes.append(len(cs_sim))
print(f"coverage of 95 % credible sets: {np.mean(covered):.1%} | size: median {np.median(sizes):.0f}, "
      f"mean {np.mean(sizes):.1f}, max {np.max(sizes)} SNPs (region has {len(region)} SNPs)")

🔍 **What just happened?** For our example the credible set contains 2 SNPs (PIPs 0.89 and 0.08), including the causal
one: instead of "a tower of 6 genome-wide significant SNPs" we can say "the causal variant is one of these 2, with 95 %
probability". Over 200 simulated studies the 95 % sets contain the causal SNP 99 % of the time — the probability
statement is **calibrated** (slightly conservative) — with a median size of 3 SNPs (mean 3.4, largest 33). Sets are large exactly when the causal SNP has near-perfect proxies: no statistical method
can separate SNPs that are always inherited together; only functional data ([Module 36](36_regulatory_genomics_deep_learning.ipynb)) or other populations with
different LD can.

**Several causal variants per region.** Real loci often harbour two or more causal variants. Single-effect regression
then fails, and stepwise conditional analysis ("add the lead SNP as a covariate and rescan") is greedy and gives no
uncertainty. **SuSiE** ("sum of single effects", Wang *et al.* 2020) writes the effect vector as a sum of $L$
single-effect vectors,

$$y = X\sum_{l=1}^{L}b_l+\epsilon,\qquad b_l=\gamma_l\,\beta_l,\quad \gamma_l\sim\text{one-hot(uniform)},\ \beta_l\sim\mathcal N(0,\sigma^2_{0l}),$$

*In words:* each of the $L$ "effects" picks exactly one SNP (the one-hot vector $\gamma_l$) and gives it an effect size.
SuSiE fits this with **iterative Bayesian stepwise selection** (IBSS): cycle over $l$, remove the other effects' current
fitted values from $y$, and fit effect $l$ to the residual with single-effect regression. Each effect yields its own
PIP vector $\alpha_l$ and its own credible set; the overall $\mathrm{PIP}_j=1-\prod_l(1-\alpha_{lj})$. Effects that find
no signal are switched off (their prior variance is estimated as 0), and credible sets whose SNPs are not in LD with
each other ("impure" sets with minimum $|r|<0.5$) are discarded. IBSS is a coordinate-ascent algorithm for a
variational approximation — the same "update one block at a time given the others" idea as the E- and M-steps of EM
([Module 13](13_mixture_models_em.ipynb)).

In [ ]:
# ── 4.5 SuSiE-lite (1/2): single-effect regression with an estimated prior variance ──
PRIOR_GRID = np.array([0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]) ** 2


def ser_with_prior(X, r, sigma2, xtx):
    """Single-effect regression whose prior variance is chosen from PRIOR_GRID by maximum marginal likelihood
    (0 = effect switched off when no grid value beats 'no effect')."""
    bhat, V = X.T @ r / xtx, sigma2 / xtx
    best = (0.0, None, None)                                     # (log marginal likelihood ratio, prior var, log BFs)
    for W in PRIOR_GRID:
        lbf = 0.5 * np.log(V / (V + W)) + 0.5 * (bhat ** 2 / V) * W / (V + W)
        lml = lbf.max() + np.log(np.mean(np.exp(lbf - lbf.max())))
        if lml > best[0]:
            best = (lml, W, lbf)
    if best[1] is None:
        return np.full(len(xtx), 1 / len(xtx)), np.zeros(len(xtx)), np.zeros(len(xtx)), 0.0
    _, W, lbf = best
    alpha = np.exp(lbf - lbf.max()); alpha /= alpha.sum()
    post_var = 1 / (1 / V + 1 / W)
    return alpha, post_var * bhat / V, post_var, W

In [ ]:
# ── 4.6 SuSiE-lite (2/2): iterative Bayesian stepwise selection and credible sets ──
def susie_lite(X, y, L=5, n_iter=100, tol=1e-4):
    """Fit y = X sum_l b_l + e by IBSS. Returns (alpha (L × J), prior variances (L,), residual variance)."""
    n, J = X.shape
    xtx = np.sum(X ** 2, axis=0)
    alpha, mu, mu2, W = np.zeros((L, J)), np.zeros((L, J)), np.zeros((L, J)), np.zeros(L)
    sigma2 = y.var()
    for _ in range(n_iter):
        alpha_old = alpha.copy()
        for l in range(L):
            r = y - X @ ((alpha * mu).sum(0) - alpha[l] * mu[l])       # residual without effect l
            alpha[l], mu[l], pv, W[l] = ser_with_prior(X, r, sigma2, xtx)
            mu2[l] = pv + mu[l] ** 2
        B = alpha * mu                                                 # posterior mean of each b_l
        erss = np.sum((y - X @ B.sum(0)) ** 2) + np.sum(xtx * alpha * mu2) - np.sum((X @ B.T) ** 2)
        sigma2 = erss / n                                              # expected residual sum of squares / n
        if np.abs(alpha - alpha_old).max() < tol:
            break
    return alpha, W, sigma2


def susie_credible_sets(X, alpha, W, level=0.95, min_purity=0.5):
    """Credible sets of the active effects, keeping only 'pure' sets (min |r| between members >= min_purity)."""
    sets = []
    for l in np.flatnonzero(W > 0):
        cs_l = credible_set(alpha[l], level)
        purity = 1.0 if len(cs_l) == 1 else np.abs(np.corrcoef(X[:, cs_l].T)).min()
        if purity >= min_purity and not any(set(cs_l) == set(c) for c in sets):
            sets.append(cs_l)
    return sets

In [ ]:
# ── 4.7 A region with THREE causal SNPs: marginal scan vs SuSiE-lite ──
rng = np.random.default_rng(SEED + 41)
true3 = np.array([12, 33, 58])
y3 = Xr[:, true3] @ (np.array([1, -1, 1]) * np.sqrt(0.025)) + rng.normal(0, 0.95, n_ld)
y3 = (y3 - y3.mean()) / y3.std()
_, _, chi2_3, p_3 = assoc_scan(Xr, y3)
alpha3, W3, _ = susie_lite(Xr, y3)
pip3 = 1 - np.prod(1 - alpha3[W3 > 0], axis=0)                 # PIP over the active effects only
print(f"active effects: {np.sum(W3 > 0)} of 5 | estimated prior SDs: {np.round(np.sqrt(W3), 2)}")
sets3 = susie_credible_sets(Xr, alpha3, W3)
top5 = np.argsort(-chi2_3)[:5]
r2_top = np.corrcoef(Xr[:, np.r_[top5, true3]].T)[:5, 5:] ** 2         # r² of each top SNP with each causal SNP
print("top 5 marginal SNPs:", top5.tolist(), "| true causal:", true3.tolist())
print("best r² of each top SNP with a causal SNP (causal SNP, r²):",
      {int(j): (int(true3[r.argmax()]), round(float(r.max()), 2)) for j, r in zip(top5, r2_top)})
for k, c in enumerate(sets3):
    print(f"credible set {k + 1}: {sorted(c.tolist())}  contains causal: {bool(np.isin(true3, c).any())}")

fig, axes = plt.subplots(2, 1, figsize=(8.5, 4.4), sharex=True)
axes[0].scatter(np.arange(len(region)), -np.log10(p_3), s=12, color="0.4")
axes[1].scatter(np.arange(len(region)), pip3, s=12, color="0.4")
for k, c in enumerate(sets3):
    axes[1].scatter(c, pip3[c], s=40, color=plt.cm.tab10(k + 1), label=f"credible set {k + 1}")
for ax in axes:
    for j in true3:
        ax.axvline(j, color="tab:red", lw=0.8, ls="--")
axes[0].set_ylabel("−log10 p"); axes[0].set_title("marginal scan (red dashes = 3 true causal SNPs, SIMULATED)")
axes[1].set_ylabel("PIP"); axes[1].set_xlabel("SNP in region"); axes[1].legend(fontsize=7, ncol=3)
axes[1].set_title("SuSiE-lite posterior inclusion probabilities and credible sets")
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.8 🧪 Ground-truth check: SuSiE-lite over 100 regions with 2–3 causal SNPs ──
t0 = time.perf_counter()
n_sets, set_ok, set_size, found, lead_ok = [], [], [], [], []
for _ in range(100):
    k = rng.integers(2, 4)
    js = rng.choice(len(region), k, replace=False)
    g = Xr[:, js] @ (rng.choice([-1, 1], k) * np.sqrt(0.025))
    y_sim = g + rng.normal(0, np.sqrt(1 - g.var()), n_ld)
    y_sim = (y_sim - y_sim.mean()) / y_sim.std()
    al, Wl, _ = susie_lite(Xr, y_sim)
    sets = susie_credible_sets(Xr, al, Wl)
    n_sets.append(len(sets))
    set_ok += [bool(np.isin(js, c).any()) for c in sets]
    set_size += [len(c) for c in sets]
    found.append(np.mean([any(j in c for c in sets) for j in js]))
    lead_ok.append(int(np.argmax(np.abs(Xr.T @ y_sim))) in js)
print(f"{time.perf_counter() - t0:.1f} s | credible sets per region: {np.mean(n_sets):.2f} (true causal: 2–3) | "
      f"sets containing a causal SNP: {np.mean(set_ok):.1%} | mean set size {np.mean(set_size):.1f} | "
      f"causal SNPs captured: {np.mean(found):.1%} | marginal lead SNP is causal: {np.mean(lead_ok):.0%}")

🔍 **What just happened?** In the three-causal example the marginal scan is misleading — three of its top five SNPs
(62, 67, 79) are proxies of causal SNP 58 ($r^2$ 0.86–0.95), and causal SNP 33 ties with its proxy 30
($-\log_{10}p=10.5$) above a crowd of 15 block-mates between 5 and 10 — while SuSiE-lite switches on exactly 3 of its 5 effects and reports 3 credible sets, each containing one causal
SNP: {58, 62}, {12} and a 4-SNP set around SNP 33. Over 100 simulated regions it reports 2.3 sets per region (truth:
2–3); 88.7 % of the reported sets contain a causal SNP, the mean set size is 2.6 SNPs, and 82 % of all causal SNPs are
captured. The marginal lead SNP is causal in only 68 % of regions.

The coverage of the sets is close to, but **below**, the nominal 95 % here (with a single causal SNP, Section 4.4, it
was at or above 95 %). That is not a bug in the arithmetic: when two effects compete for SNPs in LD, IBSS can settle in a
local optimum, and the variational approximation treats the effects as independent and becomes over-confident. The real
SuSiE adds refinements (convergence on the variational objective, estimated prior variances on a continuous scale, a
`refine` option that restarts from the current solution) — but the general lesson stands: **check the calibration of
any fine-mapping method by simulation under realistic LD**, exactly as we just did.

✏️ **Your turn.** (a) Halve the effect sizes (`np.sqrt(0.0125)`) in 4.8: how do the number of sets, their size and the
fraction of causal SNPs captured change? (b) Replace `Xr` by a matrix in which you duplicate one SNP (perfect LD):
what does the credible set of that SNP look like?

### 📄 Research connection — Statistical fine-mapping

- **What the field does.** Fine-mapping assigns each variant in a GWAS locus a posterior probability of being causal. Maller *et al.* (2012)
introduced credible sets from Bayes factors under a single-causal-variant assumption (with Wakefield's approximate
Bayes factors); CAVIAR, FINEMAP (shotgun stochastic search over configurations) and DAP-G relaxed it to several
causal variants; **SuSiE** (Wang *et al.* 2020) made multi-variant fine-mapping fast and its credible sets easy to
interpret, and SuSiE-RSS runs it from summary statistics plus an LD matrix. Schaid, Chen & Larson (2018) review
the field.
- **What this notebook builds.** Single-effect regression with Gaussian-prior Bayes factors, PIPs and credible sets, a compact IBSS
implementation with a grid-estimated prior variance and a purity filter, and the essential habit: measuring
credible-set coverage and size by simulation. Production fine-mapping usually works from **summary statistics and
a reference LD matrix**, where any mismatch between the LD of the reference panel and the study (ancestry,
imputation quality, sample QC) produces spurious signals.
- **Where the frontier is.** Functionally informed priors (PolyFun; sequence-model annotations, Module 36), **multi-ancestry fine-mapping**
that exploits different LD to shrink credible sets, fine-mapping of eQTLs and colocalisation with GWAS (coloc,
coloc-SuSiE), and calibration under misspecification (out-of-sample LD, imputation errors) are active topics;
MPRAs and CRISPR screens increasingly validate fine-mapped variants.
- **Read next.** Wang, Sarkar, Carbonetto & Stephens 2020, *A simple new approach to variable selection in regression, with
application to genetic fine mapping* (JRSS-B); Schaid, Chen & Larson 2018, *From genome-wide associations to
candidate causal variants by statistical fine-mapping* (Nat Rev Genet).

---
## 5 · Polygenic scores — and why they travel badly

> 📍 **Where we are.** Sections 3–4 asked *how much* and *where*. A **polygenic score** (PGS, also PRS for risk)
> asks *who*: predict a person's genetic value from their genotypes, $\widehat{\mathrm{PGS}}_i=\sum_j x_{ij}\hat w_j$.
> This is supervised learning (Modules 06–07 and 23) with $J\gg N$, strongly correlated features (LD) and a known
> upper bound on accuracy (the heritability).

Three classic ways to choose the weights $\hat w$:

1. **Clumping + thresholding (C+T).** Take the marginal GWAS effects $\hat\beta_j$; keep the most significant SNP in each
   LD neighbourhood ("clumping": discard SNPs with $r^2>0.1$ to an already kept, more significant SNP); keep only SNPs
   with $p<p_T$; choose $p_T$ on a validation set. Simple, interpretable, and the basis of PRSice (Choi, Mak & O'Reilly
   2020 describe the workflow).
2. **Ridge / BLUP.** All SNPs at once with a Gaussian prior — the mixed model of Section 3 with
   $\lambda=M(1-h^2)/h^2$ for $M$ standardised SNPs, where $h^2$ is the heritability *captured by those SNPs*: we
   estimate it by REML from training people (the simulation's true $h^2$ is hidden knowledge). Bayesian methods such as LDpred and PRS-CS refine the prior
   (sparse or continuous shrinkage) and work from summary statistics plus LD.
3. **Lasso.** A sparse prior (Laplace, Module 07); fitted on individual-level data in biobanks by tools such as snpnet.

**The ceiling.** No score can explain more variance than the heritability captured by the genotyped SNPs; with finite
$N$ the expected accuracy is lower still, roughly $R^2\approx h^2/\big(1+M_e/(N h^2)\big)$ for $M_e$ effectively
independent segments (Daetwyler *et al.* 2008).

**The setting.** As in real arrays, the causal variants are mostly **not genotyped**: the score must use SNPs that *tag*
them through LD. We train on 3,000 people, choose tuning parameters on 500 and test on 500.

In [ ]:
# ── 5.1 SIMULATED cohort for prediction: 4,000 people × 2,000 SNPs (50 LD blocks), 200 causal SNPs, h² = 0.5 ──
SEED_POP_A = SEED + 50
G_A, _ = simulate_ld_genotypes(np.random.default_rng(SEED_POP_A), n_samples=4000, n_snps=2000, block_length=40)
X_A = G_A.astype(float)
maf_A = np.minimum(X_A.mean(0) / 2, 1 - X_A.mean(0) / 2)
rng = np.random.default_rng(SEED + 5)
causal5 = np.sort(rng.choice(np.flatnonzero(maf_A > 0.05), 200, replace=False))
y_A, beta5, causal5 = simulate_quantitative_trait(rng, X_A, causal=causal5, h2=0.5)
g_A = X_A @ beta5                                             # true genetic values (answer key)
array = np.setdiff1d(np.flatnonzero(maf_A > 0.01), causal5)   # "array" SNPs: causal variants NOT genotyped
tr, va, te = np.arange(0, 3000), np.arange(3000, 3500), np.arange(3500, 4000)
Z_tr, mu_A, sd_A = standardize(X_A[tr][:, array])             # standardise with TRAINING statistics only
Z_va, Z_te = standardize(X_A[va][:, array], mu_A, sd_A)[0], standardize(X_A[te][:, array], mu_A, sd_A)[0]
y_tr = y_A[tr] - y_A[tr].mean()
M5 = len(array)


def r2_score_corr(pred, y):
    """Squared correlation between a score and the phenotype (the usual 'variance explained' of a PGS)."""
    return float(np.corrcoef(pred, y)[0, 1] ** 2)


print(f"{len(array)} array SNPs (causal SNPs removed) | ceiling = R² of the TRUE genetic value in the test set: "
      f"{r2_score_corr(g_A[te], y_A[te]):.3f}")
Z_1000 = standardize(X_A[:1000])[0]                           # correlations from 1,000 people are plenty
best_tag_r2 = ((Z_1000[:, causal5].T @ Z_1000[:, array] / 1000) ** 2).max(axis=1)
print(f"how well is each causal SNP tagged? best r² with an array SNP: median {np.median(best_tag_r2):.2f}, "
      f"{np.mean(best_tag_r2 > 0.8):.0%} above 0.8")

In [ ]:
# ── 5.2 Clumping + thresholding ──
b_tr, _, _, p_tr = assoc_scan(Z_tr, y_tr)
R_tr = np.corrcoef(Z_tr.T)


def clump(p, R, r2_max=0.1):
    """Greedy LD clumping: walk SNPs from most to least significant, keep one, drop its LD partners (r² > r2_max)."""
    keep, removed = [], np.zeros(len(p), bool)
    for j in np.argsort(p):
        if not removed[j]:
            keep.append(j)
            removed |= R[j] ** 2 > r2_max
    return np.array(keep)


clumped = clump(p_tr, R_tr)
rows, best_ct = [], (-1, None)
for p_T in [5e-8, 1e-5, 1e-3, 1e-2, 0.05, 0.5, 1.0]:
    sel = clumped[p_tr[clumped] < p_T]
    r2_va = r2_score_corr(Z_va[:, sel] @ b_tr[sel], y_A[va]) if len(sel) else 0.0
    rows.append({"p threshold": p_T, "SNPs in score": len(sel), "validation R²": round(r2_va, 3)})
    if r2_va > best_ct[0]:
        best_ct = (r2_va, sel)
sel_ct = best_ct[1]
r2_ct = r2_score_corr(Z_te[:, sel_ct] @ b_tr[sel_ct], y_A[te])
print(f"{len(clumped)} clumps; chosen on validation: {len(sel_ct)} SNPs → TEST R² = {r2_ct:.3f}")
pd.DataFrame(rows).set_index("p threshold")

In [ ]:
# ── 5.3 SNP heritability of the ARRAY SNPs by REML (Section 3), estimated from 1,500 training people ──
t0 = time.perf_counter()
sub5 = tr[:1500]                                              # a subset keeps the N × N eigendecomposition small
K5 = Z_tr[sub5] @ Z_tr[sub5].T / M5
s5, U5 = np.linalg.eigh(K5)
h2_array, _ = fit_lmm(U5.T @ y_tr[sub5], U5.T @ np.ones((len(sub5), 1)), np.clip(s5, 0, None))
del K5, U5
lam_blup = M5 * (1 - h2_array) / h2_array                     # λ = M (1 − ĥ²) / ĥ²: no peeking at the true h²
print(f"REML h² of the array SNPs (training people only): {h2_array:.3f} → λ = M(1 − h²)/h² = {lam_blup:.0f} "
      f"({time.perf_counter() - t0:.1f} s)")

In [ ]:
# ── 5.4 Ridge/BLUP with that λ, a validation grid of λ, and the lasso ──
A_tr = Z_tr.T @ Z_tr
c_tr = Z_tr.T @ y_tr
w_blup = np.linalg.solve(A_tr + lam_blup * np.eye(M5), c_tr)
ridge_rows = []
for lam in [30, 100, 300, lam_blup, 3000, 10000]:
    w = np.linalg.solve(A_tr + lam * np.eye(M5), c_tr)
    ridge_rows.append((lam, r2_score_corr(Z_va @ w, y_A[va])))
lam_cv = max(ridge_rows, key=lambda t: t[1])[0]
print("ridge validation R² by λ:", [(round(l), round(r, 3)) for l, r in ridge_rows])
r2_blup = r2_score_corr(Z_te @ w_blup, y_A[te])
print(f"BLUP with λ = M(1−ĥ²)/ĥ² = {lam_blup:.0f}: TEST R² = {r2_blup:.3f} (validation-best λ in the grid = {lam_cv:.0f})")

t0 = time.perf_counter()
lasso_rows = []
for a in [0.003, 0.01, 0.03]:                                 # scikit-learn's alpha = λ / (2N) (Module 07)
    m = Lasso(alpha=a, max_iter=5000).fit(Z_tr, y_tr)
    lasso_rows.append((a, r2_score_corr(m.predict(Z_va), y_A[va]), m))
a_best, _, lasso_best = max(lasso_rows, key=lambda t: t[1])
r2_lasso = r2_score_corr(lasso_best.predict(Z_te), y_A[te])
print(f"lasso ({time.perf_counter() - t0:.1f} s): alpha = {a_best} chosen on validation, "
      f"{np.sum(lasso_best.coef_ != 0)} non-zero weights → TEST R² = {r2_lasso:.3f}")

fig, ax = plt.subplots(figsize=(6.2, 3.0))
names = ["C+T", "ridge / BLUP", "lasso"]
ax.bar(names, [r2_ct, r2_blup, r2_lasso], color=["tab:gray", "tab:blue", "tab:green"])
ax.axhline(r2_score_corr(g_A[te], y_A[te]), color="tab:red", ls="--", label="ceiling: true genetic value")
ax.set_ylabel("test R²"); ax.set_ylim(0, 0.6); ax.legend(fontsize=7)
ax.set_title("polygenic scores on held-out people (SIMULATED population A)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Although none of the 200 causal SNPs is on the "array", every score predicts the trait in new
people, because array SNPs tag the causal ones (median best $r^2=0.94$; 92 % of causal SNPs have a tag with
$r^2>0.8$). C+T reaches a test $R^2$ of 0.33. For ridge/BLUP, the mixed model of Section 3 — fitted to 1,500 training
people only — estimates that the array SNPs capture $\hat h^2=0.47$ (a little below the simulated 0.5: tagging is
imperfect and the estimate has an SE of a few hundredths); the implied $\lambda=M(1-\hat h^2)/\hat h^2=1997$, chosen without
looking at any validation data, reaches a test $R^2$ of 0.40 and also happens to be the best λ of the validation grid.
The lasso (370 non-zero weights) reaches 0.41. All stay below the ceiling of 0.46 set by the true genetic values. The whole-genome methods beat C+T because they use all
tags jointly instead of one SNP per clump; ridge and lasso are close because this trait sits between Module 07's two
regimes — 200 causal variants is sparse relative to 1,770 SNPs, but each signal is spread over several correlated tags.

🧪 **Ground truth.** The ceiling is computed from the *true* genetic values, which no real study has; in practice the
reference is the SNP heritability from Section 3.

⚠️ **Selection bias ([Module 23](23_model_selection.ipynb)).** We chose $p_T$, λ and α on the validation people and report $R^2$ on *different* test
people. Reporting the validation $R^2$ of the best threshold — a common practice in early PGS papers — is optimistic.

**Portability.** Now the question of Martin *et al.* (2019): how well does a score trained in population A predict in
a population B whose haplotypes differ? We simulate B as a **mosaic**: in a fraction of the LD blocks B carries the same
ancestral haplotypes as A (we call the simulator again with A's seed — the ancestral haplotypes are drawn before the
individuals, so they are identical), in the remaining blocks it carries an unrelated set (a different seed). The causal
variants and their effects are **the same in both populations**; only LD differs.

In [ ]:
# ── 5.5 🧪 Portability: population B shares a fraction of A's haplotype blocks; causal effects are identical ──
G_Bsame, _ = simulate_ld_genotypes(np.random.default_rng(SEED_POP_A), n_samples=1000, n_snps=2000, block_length=40)
G_Bnew, _ = simulate_ld_genotypes(np.random.default_rng(SEED + 51), n_samples=1000, n_snps=2000, block_length=40)
print("allele-frequency correlation with population A: same-seed draw",
      round(float(np.corrcoef(G_Bsame.mean(0), X_A.mean(0))[0, 1]), 3), "| new-seed draw",
      round(float(np.corrcoef(G_Bnew.mean(0), X_A.mean(0))[0, 1]), 3))
block5 = np.arange(2000) // 40
block_order = np.random.default_rng(SEED + 52).permutation(50)     # nested sets of shared blocks
rows = []
for share in [1.0, 0.75, 0.5, 0.25, 0.0]:
    shared = np.isin(block5, block_order[:int(round(share * 50))])
    X_B = np.where(shared[None, :], G_Bsame, G_Bnew).astype(float)
    g_B = X_B @ beta5
    y_B = g_B + np.random.default_rng(SEED + 53).normal(0, np.sqrt(0.5), 1000)   # same noise variance as in A
    Z_B = standardize(X_B[:, array], mu_A, sd_A)[0]
    rows.append({"shared blocks": share, "ceiling (true g)": r2_score_corr(g_B, y_B),
                 "ridge/BLUP": r2_score_corr(Z_B @ w_blup, y_B),
                 "C+T": r2_score_corr(Z_B[:, sel_ct] @ b_tr[sel_ct], y_B)})
port = pd.DataFrame(rows).set_index("shared blocks").round(3)
port["BLUP / ceiling"] = (port["ridge/BLUP"] / port["ceiling (true g)"]).round(2)

fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.plot(port.index, port["ceiling (true g)"], "k--o", ms=4, label="ceiling: score built from the causal variants")
ax.plot(port.index, port["ridge/BLUP"], "o-", color="tab:blue", label="ridge/BLUP score trained in A")
ax.plot(port.index, port["C+T"], "s-", color="tab:gray", label="C+T score trained in A")
ax.set_xlabel("fraction of LD blocks with A's haplotypes"); ax.set_ylabel("R² in population B")
ax.invert_xaxis(); ax.legend(fontsize=7); ax.set_title("polygenic scores lose accuracy when LD differs (SIMULATED)")
plt.tight_layout(); plt.show()
port

🔍 **What just happened?** The control row (all blocks shared) is simply a new sample from population A: the BLUP score
reaches $R^2=0.46$ there (83 % of that population's ceiling). As the fraction of shared haplotype blocks falls,
accuracy collapses — 0.22 with half of the blocks shared, 0.006 with none — although **the biology is identical**: same
causal variants, same effects (the dashed ceiling of a score built from the causal variants themselves stays between
0.42 and 0.55; it varies only because allele frequencies, and hence genetic variance, differ). C+T, which relies on
single tag SNPs, degrades even faster. The tag SNPs
that carried the signal in A are correlated with different causal alleles, or with none, in B.

Our two-population model is crude — real populations share most haplotypes at different frequencies and differ in LD
block lengths (shorter in African-ancestry populations, which carry the most genetic diversity) — so the size of the drop
is not a prediction; the mechanism is. Martin *et al.* (2019) found that scores trained in European-ancestry cohorts were
roughly two to four and a half times less accurate in East Asian and African-ancestry individuals. Differences in allele
frequencies, in environments and in gene–environment interactions add to the LD effect.

✏️ **Your turn.** Make the causal variants part of the array (`array = np.flatnonzero(maf_A > 0.01)`) and rerun 5.1–5.5.
Does the score port better? Why does fine-mapping (Section 4) help portability?

---
## 6 · Mendelian randomisation: genetics as a natural experiment

> 📍 **Where we are.** Everything so far was about *genetic* effects. Mendelian randomisation (MR) uses genetic variants
> as tools to learn about the causal effect of a *non-genetic* exposure (LDL cholesterol, body-mass index, alcohol) on an
> outcome (heart disease). It is the instrumental-variable method of econometrics applied to genotypes; [Module 41](41_perturbation_causal_inference.ipynb)
> treats instrumental variables and causal inference in general.

**Why observational regression fails.** Regress heart disease on LDL across people and you get an association, but
people with high LDL also differ in diet, exercise, smoking and medication — **confounders** $U$ that affect both. The
regression slope mixes the causal effect with confounding, and no list of measured covariates is ever complete.

**The instrument.** Alleles are assigned at conception, independently of lifestyle. A variant $Z$ that raises LDL is
therefore like a randomly assigned, lifelong dose of extra LDL. Three assumptions make $Z$ a valid **instrument**:

1. **Relevance:** $Z$ is associated with the exposure $X$.
2. **Independence:** $Z$ is independent of the confounders $U$ (true by Mendel's laws within a population — but
   population structure can break it, Section 2).
3. **Exclusion restriction:** $Z$ affects the outcome $Y$ *only through* $X$ — no **horizontal pleiotropy**.

> 🔗 **Builds on — Modules [15](15_bayesian_networks_representation.ipynb)–[16](16_bayesian_networks_independence.ipynb)
> (Bayesian networks and d-separation).** Draw the causal diagram $Z\to X\to Y$, $U\to X$, $U\to Y$ (next cell).
> Assumptions 2 and 3 are statements about this graph: $Z$ has no edge from $U$ and no edge to $Y$, so every path from
> $Z$ to $Y$ passes through $X$, and conditioning on nothing, $Z$ and $U$ are d-separated. Observational regression of
> $Y$ on $X$ is biased because of the open back-door path $X\leftarrow U\to Y$; the instrument sidesteps it.

🧮 **Math refresher — the Wald ratio.** With linear effects, $X=\gamma Z+U+\epsilon_X$ and $Y=\theta X+U+\epsilon_Y$. Then
the SNP–outcome association is $\beta_{ZY}=\theta\,\gamma=\theta\,\beta_{ZX}$, so

$$\hat\theta_{\text{Wald}}=\frac{\hat\beta_{ZY}}{\hat\beta_{ZX}},\qquad
\hat\theta_{\text{IVW}}=\frac{\sum_j \hat\beta_{ZX,j}\hat\beta_{ZY,j}\,\sigma_{Y,j}^{-2}}{\sum_j \hat\beta_{ZX,j}^2\,\sigma_{Y,j}^{-2}} .$$

*In words:* the confounder never enters $\beta_{ZY}$ or $\beta_{ZX}$ because $Z$ is independent of $U$; the ratio of the
two associations is the causal effect. With many SNPs, the **inverse-variance-weighted** (IVW) estimator combines their
ratios — it is the slope of a weighted regression of $\hat\beta_{ZY}$ on $\hat\beta_{ZX}$ *through the origin*. Both
associations can come from **different studies** (two-sample MR), so only GWAS summary statistics are needed.

In [ ]:
# ── 6.1 The causal diagram and a SIMULATED two-sample MR study ──
fig, ax = plt.subplots(figsize=(5.2, 3.0))
draw_graph([("Z", "X"), ("X", "Y"), ("U", "X"), ("U", "Y"), ("Z", "Y")],
           {"Z": (0.05, 0.15), "X": (0.45, 0.6), "Y": (0.9, 0.15), "U": (0.9, 0.95)}, ax=ax, highlight={"U"},
           edge_labels=["γ (relevance)", "θ = causal effect", "", "", "α = pleiotropy (must be 0)"])
ax.set_title("instrument Z (SNPs) → exposure X → outcome Y; U = unmeasured confounder", fontsize=9)
plt.tight_layout(); plt.show()


def simulate_mr_study(rng, n, gamma, alpha, theta=0.3):
    """One cohort: SNP genotypes Z (n × J), exposure x and outcome y with an unmeasured confounder u."""
    Z, _, _ = simulate_genotypes(rng, n, len(gamma))
    Z = Z.astype(float)
    u = rng.normal(size=n)
    x = Z @ gamma + u + rng.normal(size=n)                    # exposure (e.g. LDL cholesterol, SD units)
    y = theta * x + Z @ alpha + u + rng.normal(size=n)        # outcome (liability to disease, SD units)
    return Z, x, y


rng = np.random.default_rng(SEED + 6)
N_IV, THETA = 30, 0.3
gamma = rng.uniform(0.03, 0.2, N_IV)                          # SNP → exposure effects
scenarios = {"valid instruments": np.zeros(N_IV),
             "balanced pleiotropy": rng.normal(0, 0.03, N_IV),
             "directional pleiotropy": np.where(np.arange(N_IV) < 10, rng.uniform(0.02, 0.08, N_IV), 0.0)}
print(f"30 SNP instruments, true causal effect theta = {THETA}; two independent samples of 10,000 people")

**When the exclusion restriction fails.** If some SNPs also affect $Y$ directly ($\alpha_j\neq0$), then
$\beta_{ZY,j}=\theta\beta_{ZX,j}+\alpha_j$ and IVW is biased whenever the $\alpha_j$ do not average out. Two robust
estimators:

- **MR-Egger** (Bowden, Davey Smith & Burgess 2015): regress $\hat\beta_{ZY}$ on $\hat\beta_{ZX}$ *with an intercept*
  (after orienting each SNP so that $\hat\beta_{ZX}>0$). The intercept estimates the average pleiotropic effect, the slope
  the causal effect — valid if pleiotropy is independent of instrument strength (the "InSIDE" assumption), but noisy.
- **Weighted median** (Bowden *et al.* 2016): the weighted median of the per-SNP ratios; consistent if at least half of
  the weight comes from valid instruments.

In [ ]:
# ── 6.2 Estimators from summary statistics ──
def mr_ivw(bx, by, se_y):
    """IVW estimate with a multiplicative random-effects standard error (inflated if SNPs disagree)."""
    w = bx ** 2 / se_y ** 2
    est = np.sum(bx * by / se_y ** 2) / np.sum(w)
    q = np.sum((by - est * bx) ** 2 / se_y ** 2) / (len(bx) - 1)    # Cochran's Q / d.f. (heterogeneity)
    return est, np.sqrt(max(q, 1.0) / np.sum(w))


def mr_egger(bx, by, se_y):
    """MR-Egger: weighted regression of by on bx with intercept, SNPs oriented so that bx > 0.
    Returns (slope, slope SE, intercept, intercept SE)."""
    s = np.sign(bx)
    bx, by = bx * s, by * s
    W = 1 / se_y ** 2
    A = np.column_stack([np.ones_like(bx), bx])
    coef = np.linalg.solve(A.T @ (W[:, None] * A), A.T @ (W * by))
    phi = np.sum(W * (by - A @ coef) ** 2) / (len(bx) - 2)
    cov = np.linalg.inv(A.T @ (W[:, None] * A)) * max(phi, 1.0)
    return coef[1], np.sqrt(cov[1, 1]), coef[0], np.sqrt(cov[0, 0])


def mr_weighted_median(bx, by, se_y):
    """Weighted median of the Wald ratios with first-order weights (bx / se_y)²."""
    ratio, w = by / bx, (bx / se_y) ** 2
    order = np.argsort(ratio)
    cw = np.cumsum(w[order]) / w.sum() - 0.5 * w[order] / w.sum()
    return float(np.interp(0.5, cw, ratio[order]))

In [ ]:
# ── 6.3 One two-sample MR study per scenario: observational slope vs IVW, MR-Egger and weighted median ──
rows, example = [], None
for name, alpha in scenarios.items():
    Z1s, x1s, _ = simulate_mr_study(rng, 10_000, gamma, alpha, THETA)     # sample 1: SNP–exposure GWAS
    Z2s, x2s, y2s = simulate_mr_study(rng, 10_000, gamma, alpha, THETA)   # sample 2: SNP–outcome GWAS
    bx, sx, _, _ = assoc_scan(Z1s, x1s)
    by, sy, _, _ = assoc_scan(Z2s, y2s)
    ivw, ivw_se = mr_ivw(bx, by, sy)
    eg, eg_se, eg_int, eg_int_se = mr_egger(bx, by, sy)
    rows.append({"scenario": name, "observational slope": np.polyfit(x2s, y2s, 1)[0], "IVW": ivw, "IVW SE": ivw_se,
                 "MR-Egger": eg, "Egger SE": eg_se, "Egger intercept": eg_int, "weighted median": mr_weighted_median(bx, by, sy),
                 "mean F-statistic": np.mean((bx / sx) ** 2)})
    if name == "directional pleiotropy":
        example = (bx, by, sy, ivw, eg, eg_int)
pd.DataFrame(rows).set_index("scenario").round(3)

In [ ]:
# ── 6.4 The MR scatter plot for the directional-pleiotropy scenario ──
bx, by, sy, ivw, eg, eg_int = example
s = np.sign(bx)
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.errorbar(bx * s, by * s, yerr=1.96 * sy, fmt="o", ms=4, color="tab:gray", ecolor="0.8",
            label="SNPs (±1.96 SE); first 10 are pleiotropic")
ax.scatter((bx * s)[:10], (by * s)[:10], s=22, color="tab:orange", zorder=3)
xx = np.linspace(0, (bx * s).max() * 1.05, 10)
ax.plot(xx, THETA * xx, "k--", label=f"true causal effect θ = {THETA}")
ax.plot(xx, ivw * xx, color="tab:blue", label=f"IVW (through origin) = {ivw:.2f}")
ax.plot(xx, eg_int + eg * xx, color="tab:green", label=f"MR-Egger: slope {eg:.2f}, intercept {eg_int:.3f}")
ax.set_xlabel("SNP → exposure effect  β_ZX"); ax.set_ylabel("SNP → outcome effect  β_ZY")
ax.set_title("two-sample MR with directional pleiotropy (SIMULATED)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.5 🧪 Ground-truth check: bias and 95 % CI coverage over 100 replicate studies per scenario ──
t0 = time.perf_counter()
rows = []
for name, alpha in scenarios.items():
    res = {"obs": [], "ivw": [], "ivw_cov": [], "egger": [], "egger_cov": [], "wm": []}
    for _ in range(100):
        Z1s, x1s, _ = simulate_mr_study(rng, 10_000, gamma, alpha, THETA)
        Z2s, x2s, y2s = simulate_mr_study(rng, 10_000, gamma, alpha, THETA)
        bx, _, _, _ = assoc_scan(Z1s, x1s)
        by, sy, _, _ = assoc_scan(Z2s, y2s)
        e, se = mr_ivw(bx, by, sy)
        eg, eg_se, _, _ = mr_egger(bx, by, sy)
        res["obs"].append(np.polyfit(x2s, y2s, 1)[0])
        res["ivw"].append(e); res["ivw_cov"].append(abs(e - THETA) < 1.96 * se)
        res["egger"].append(eg); res["egger_cov"].append(abs(eg - THETA) < 1.96 * eg_se)
        res["wm"].append(mr_weighted_median(bx, by, sy))
    rows.append({"scenario": name, "observational": np.mean(res["obs"]), "IVW mean": np.mean(res["ivw"]),
                 "IVW coverage": np.mean(res["ivw_cov"]), "Egger mean": np.mean(res["egger"]),
                 "Egger SD": np.std(res["egger"]), "Egger coverage": np.mean(res["egger_cov"]),
                 "weighted median mean": np.mean(res["wm"])})
print(f"{time.perf_counter() - t0:.1f} s; true theta = {THETA}")
pd.DataFrame(rows).set_index("scenario").round(3)

🔍 **What just happened?** The observational slope is about 0.77 in every scenario — two and a half times the true
0.3, because the confounder raises both. With **valid instruments**, IVW recovers the truth (mean 0.291 over 100
studies, 95 % intervals covering 0.3 in 97 %; the slight shortfall is the weak-instrument bias of two-sample MR towards
zero). **Balanced** pleiotropy adds noise that the random-effects SE absorbs (coverage 93 %), although our 30
pleiotropic effects do not average exactly to zero, so all estimators drift to ≈ 0.34. **Directional** pleiotropy (a
third of the SNPs push the outcome up on their own) biases IVW (mean 0.420, coverage 49 %) and the weighted median
(0.365 — a third of the instruments are invalid, not far from its breakdown point of one half). MR-Egger stays close to
the truth (0.264, coverage 95 %) and its intercept flags the pleiotropy, but its spread (SD 0.13) is about twice the
IVW standard error (0.04–0.07): robustness costs precision, and weak-instrument dilution hits Egger hardest.

The scatter plot is how MR results are shown in papers: each SNP is a point, the IVW line is forced through the origin,
the Egger line is not. The F-statistics (mean ≈ 25–28) say the instruments are not weak; weak instruments ($F<10$) bias
two-sample MR towards zero.

⚠️ **MR is only as good as its assumptions.** Pleiotropy is common in real biology (a variant in a lipid gene may act on
several lipid fractions); population structure, assortative mating and dynastic effects can violate independence;
"exposure" usually means a lifelong genetic shift, not the effect of a drug given at age 60; and selection into a biobank
can create collider bias ([Module 16](16_bayesian_networks_independence.ipynb)). Sensitivity analyses with several estimators, and triangulation with other
evidence, are standard.

✏️ **Your turn.** Make pleiotropy *correlated with instrument strength* (violating InSIDE), e.g. `alpha = 0.3 * gamma`
for the first 10 SNPs. Which estimator is now biased? Then make the instruments weak (`gamma = rng.uniform(0.005, 0.02,
N_IV)`): what happens to the mean F-statistic and to IVW?

### 📄 Research connection — Polygenic scores, equity and Mendelian randomisation

- **What the field does.** Polygenic scores moved from C+T (Purcell *et al.* 2009; PRSice — Choi, Mak & O'Reilly 2020 give a tutorial) to
Bayesian shrinkage from summary statistics (LDpred/LDpred2, PRS-CS, SBayesR) and penalised regression on
biobank data. Martin *et al.* (2019) documented the large loss of accuracy in non-European-ancestry groups and its
roots in Eurocentric discovery cohorts. Mendelian randomisation was framed by Davey Smith & Ebrahim (2003); with
GWAS summary statistics it became two-sample MR with IVW (Burgess, Butterworth & Thompson 2013), MR-Egger (Bowden
*et al.* 2015), median and mode estimators, and platforms such as MR-Base/TwoSampleMR.
- **What this notebook builds.** C+T, ridge/BLUP with λ from the REML heritability, the lasso, a true-genetic-value ceiling, a controlled
portability experiment, and IVW, MR-Egger and weighted-median estimators with bias and coverage measured over
replicate studies. Real pipelines add QC, ancestry-matched LD references and absolute-risk calibration.
- **Where the frontier is.** Multi-ancestry and fine-mapping-informed scores and diverse biobanks (*All of Us*, H3Africa, Biobank Japan, Million
Veteran Program) aim to close the portability gap; the clinical utility of PGS remains debated. In MR: multivariable
MR, robust methods for many invalid instruments, within-family MR, molecular exposures (proteins, expression) for
drug-target validation, and selection bias in biobanks.
- **Read next.** Martin *et al.* 2019, *Clinical use of current polygenic risk scores may exacerbate health disparities*;
Choi, Mak & O'Reilly 2020, *Tutorial: a guide to performing polygenic risk score analyses*;
Bowden, Davey Smith & Burgess 2015, *Mendelian randomization with invalid instruments: effect estimation and bias
detection through Egger regression*.

---
## 7 · Working with summary statistics: LD-score regression

> 📍 **Where we are.** Sections 1–3 showed that $\lambda_{GC}>1$ can mean confounding (Section 2) *or* many true tiny
> effects (the ✏️ of Section 1). Most GWAS results are shared only as **summary statistics** (one $\hat\beta$, SE and $p$
> per SNP), so a method that tells the two apart *from summary statistics alone* is extremely useful.

**The idea of Bulik-Sullivan *et al.* (2015).** Under a polygenic model in which every standardised SNP has a small
random effect with variance $h^2/M$, a SNP's expected $\chi^2$ grows with how much genetic signal it *tags* — its **LD
score** $\ell_j=\sum_k r^2_{jk}$ (the sum of squared correlations with all SNPs nearby, including itself). Confounding by
stratification or relatedness, by contrast, inflates all SNPs about equally, regardless of LD:

$$\mathbb E\big[\chi^2_j\big]\;=\;1\;+\;\frac{N\,h^2}{M}\,\ell_j\;+\;N a .$$

*In words:* regress the $\chi^2$ statistics on the LD scores. The **slope** measures polygenic signal (it gives $h^2$
from summary statistics), the **intercept** measures inflation that does not depend on LD — confounding — and should be 1
in a clean study. A SNP in a region of extensive LD has a high $\ell_j$ and "collects" the effects of many neighbours.

We need many independent LD blocks for a stable regression, so we simulate a new cohort with 6,000 SNPs in 150 blocks,
and traits whose effects follow the LDSC model (equal variance per *standardised* SNP).

In [ ]:
# ── 7.1 LD scores of a SIMULATED cohort: 1,500 people × 6,000 SNPs (150 LD blocks) ──
t0 = time.perf_counter()
G7, _ = simulate_ld_genotypes(np.random.default_rng(SEED + 7), n_samples=1500, n_snps=6000, block_length=40)
X7 = G7.astype(float)
maf7 = np.minimum(X7.mean(0) / 2, 1 - X7.mean(0) / 2)
Z7 = standardize(X7[:, maf7 > 0.01])[0]
N7, M7 = Z7.shape
del X7


def ld_scores(Z, window=60, chunk=500):
    """l_j = sum of bias-corrected r² with SNPs within ±window positions (computed in chunks to save memory)."""
    n, M = Z.shape
    ell = np.zeros(M)
    for start in range(0, M, chunk):
        stop = min(M, start + chunk)
        lo, hi = max(0, start - window), min(M, stop + window)
        r2 = (Z[:, start:stop].T @ Z[:, lo:hi] / n) ** 2
        r2 -= (1 - r2) / (n - 2)                               # remove the upward bias of sample r²
        dist = np.abs(np.arange(start, stop)[:, None] - np.arange(lo, hi)[None, :])
        ell[start:stop] = np.sum(r2 * (dist <= window), axis=1)
    return ell


ell7 = ld_scores(Z7)
print(f"{M7} SNPs, LD scores {ell7.min():.1f}–{ell7.max():.1f} (median {np.median(ell7):.1f}) in "
      f"{time.perf_counter() - t0:.1f} s | correlation of LD score with MAF: {np.corrcoef(maf7[maf7 > 0.01], ell7)[0, 1]:.2f}")


def chi2_from_z(Z, y):
    """Fast marginal chi² statistics for standardised genotypes (no covariates)."""
    yc = y - y.mean()
    Zc = Z - Z.mean(0)
    sxx = np.sum(Zc ** 2, axis=0)
    b = Zc.T @ yc / sxx
    return b ** 2 * sxx / ((yc @ yc - b ** 2 * sxx) / (len(y) - 2))


def ldsc(chi2, ell, N, M):
    """Unweighted LD-score regression: returns (intercept, h² = slope · M / N)."""
    slope, intercept = np.polyfit(ell, chi2, 1)
    return intercept, slope * M / N

In [ ]:
# ── 7.2 🧪 Ground-truth check: 15 replicate GWAS without and with confounding by relatives ──
def ldsc_study(rng, n_pairs=0, h2=0.3, shared_env=0.3):
    """One GWAS of a polygenic trait (every standardised SNP has effect ~ N(0, h2/M)). n_pairs people appear twice
    (identical twins) and the members of each family share an environment of variance shared_env.
    Returns (chi2 per SNP, total sample size)."""
    g = Z7 @ rng.normal(0, np.sqrt(h2 / M7), M7)
    fam = np.r_[np.arange(N7), np.arange(n_pairs)]            # twin = duplicated genotype row
    env = rng.normal(0, np.sqrt(shared_env), N7)[fam]          # one environment per family
    y = g[fam] + env + rng.normal(0, np.sqrt(1 - h2 - shared_env), len(fam))
    return chi2_from_z(Z7[fam], y), len(fam)


rng = np.random.default_rng(SEED + 70)
t0 = time.perf_counter()
scen7 = {"1,500 unrelated people": 0, "+ 400 identical twins sharing an environment": 400}
chi2_store, rows = {}, []
for name, pairs in scen7.items():
    chi2_store[name], est = [], []
    for _ in range(15):
        chi2_r, n_r = ldsc_study(rng, n_pairs=pairs)
        chi2_store[name].append(chi2_r)
        est.append((*ldsc(chi2_r, ell7, n_r, M7), lambda_gc(chi2_r)))     # (intercept, h², lambda_GC)
    est = np.array(est)
    rows.append({"scenario": name, "lambda_GC": est[:, 2].mean(), "intercept mean": est[:, 0].mean(),
                 "intercept SD": est[:, 0].std(ddof=1), "h² mean": est[:, 1].mean(), "h² SD": est[:, 1].std(ddof=1)})
print(f"{time.perf_counter() - t0:.1f} s; true h² = 0.3; the intercept should be 1 without confounding")
pd.DataFrame(rows).set_index("scenario").round(3)

In [ ]:
# ── 7.3 χ² against LD score, averaged over the 15 replicates (thin lines = single-study fits) ──
bins = np.quantile(ell7, np.linspace(0, 1, 21))
which = np.clip(np.digitize(ell7, bins) - 1, 0, 19)
mid = np.array([ell7[which == k].mean() for k in range(20)])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, (name, pairs) in zip(axes, scen7.items()):
    n_tot = N7 + pairs
    for chi2_r in chi2_store[name]:
        ic_r, h_r = ldsc(chi2_r, ell7, n_tot, M7)
        ax.plot(mid, ic_r + h_r * n_tot / M7 * mid, color="tab:red", lw=0.5, alpha=0.3)
    chi2_mean = np.mean(chi2_store[name], axis=0)
    ic, h2_ = ldsc(chi2_mean, ell7, n_tot, M7)
    ax.plot(mid, [chi2_mean[which == k].mean() for k in range(20)], "o", color="tab:blue", ms=4,
            label="mean χ² per LD-score bin")
    ax.plot(mid, ic + h2_ * n_tot / M7 * mid, color="tab:red", lw=2, label=f"average fit: intercept {ic:.2f}, h² {h2_:.2f}")
    ax.axhline(1, color="0.5", ls=":", lw=0.8)
    ax.set_xlabel("LD score ℓ"); ax.set_title(name); ax.legend(fontsize=7, loc="upper left")
axes[0].set_ylabel("χ²")
plt.suptitle("LD-score regression on 15 SIMULATED GWAS per scenario (true h² = 0.3)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the clean studies $\lambda_{GC}=1.61$ on average — clearly above 1 — although there is
**no confounding at all**: the inflation is pure polygenic signal, and the binned $\chi^2$ rise linearly with LD score.
LD-score regression puts it where it belongs: over 15 replicates the intercept is 1.03 ± 0.16 and $\hat h^2=0.28\pm0.08$
(truth 0.3). Adding 400 identical twins who share an environment — a caricature of the relatives that every biobank
contains — raises the **intercept** to 1.30 on average: the twins' residuals are correlated, so every SNP's test is
over-confident by a similar factor, whatever its LD, while $\hat h^2$ stays near the truth (0.26 ± 0.12). A careless
analyst looking only at $\lambda_{GC}$ (1.93) could not tell how much of the inflation is confounding. Single estimates
are noisy here (thin red lines; intercept SD 0.16–0.26) because we have only 150 LD blocks; real LDSC uses about a
million SNPs and a block jackknife for standard errors.

⚠️ **LDSC is a model, not a theorem.** It assumes that per-SNP heritability does not depend on LD score. In our simulator
LD score is strongly correlated with allele frequency (r = 0.71), so a trait whose effects are larger for common SNPs — e.g.
effects drawn on the 0/1/2 scale, as `simulate_quantitative_trait` does — violates the assumption and biases both intercept
and slope. Stratified LDSC and related methods model such dependence explicitly.

✏️ **Your turn.** Make per-SNP heritability depend on allele frequency: in `ldsc_study`, draw the effects with standard
deviation proportional to `np.sqrt(maf7[maf7 > 0.01])` (rescaled so that the total $h^2$ stays 0.3). Over 10 replicates,
what happens to the intercept and to $\hat h^2$, and why?

---
## 8 · 🧭 From toy to real data

> 📍 **Where we are.** Every algorithm in this notebook ran on a few thousand simulated people in seconds. Here is what
> changes with real biobanks. (Reading only — nothing is downloaded.) Good overviews of the field are Visscher *et al.*
> (2017) and Uffelmann *et al.* (2021).

**Data and resources.**
- [UK Biobank](https://www.ukbiobank.ac.uk/) (≈ 500,000 participants; arrays, imputation, exome and whole-genome
  sequencing, imaging, health records; Bycroft *et al.* 2018), [FinnGen](https://www.finngen.fi/en),
  [All of Us](https://allofus.nih.gov/), [Biobank Japan](https://biobankjp.org/en/), the Million Veteran Program —
  all with controlled access after an application.
- Summary statistics: the [GWAS Catalog](https://www.ebi.ac.uk/gwas/) and consortium portals; LD reference panels from
  the [1000 Genomes Project](https://www.internationalgenome.org/) and gnomAD.

**Production tools (what replaces our code).**

| Task | This notebook | Production tools |
|---|---|---|
| QC, association | `assoc_scan` | [PLINK 2](https://www.cog-genomics.org/plink/2.0/), REGENIE, SAIGE (binary traits), BOLT-LMM, fastGWA |
| Heritability, GRM | EMMA/FaST-LMM-style REML | GCTA, BOLT-REML, LDSC, LDAK |
| Fine-mapping | SER, SuSiE-lite | susieR / SuSiE-RSS, FINEMAP, PolyFun, coloc |
| Polygenic scores | C+T, ridge, lasso | PRSice-2, LDpred2, PRS-CS(x), SBayesRC, snpnet; the PGS Catalog for published scores |
| Mendelian randomisation | IVW, Egger, weighted median | TwoSampleMR / MR-Base, MendelianRandomization (R) |

**Compute.** A GRM for 500,000 people would have $2.5\times10^{11}$ entries; its eigendecomposition is out of the
question, so BOLT-LMM (conjugate gradients) and REGENIE (ridge on SNP blocks) scale roughly linearly in $N$. A biobank
GWAS takes hours on a cluster or cloud platform that brings the computation to the data.

**The first three things that go wrong.**
1. **Quality control and hidden structure.** Array/sequencing batch effects, sample swaps, relatives, duplicates and
   poorly imputed variants produce hits that look real: inspect QQ plots, check the LDSC intercept, replicate.
2. **Mismatched LD and ancestry.** Summary-statistic methods (SuSiE-RSS, PRS-CS, LDSC) need a reference LD matrix; if
   it differs from the study, credible sets and scores become wrong — sometimes confidently wrong.
3. **Binary traits and ascertainment.** Linear models on rare 0/1 outcomes are miscalibrated (SAIGE and REGENIE use
   saddle-point approximations), $h^2$ must be put on the liability scale, and volunteers are healthier and wealthier
   than the population (selection bias).

**Ethics and responsible use.** Genotypes identify people and their relatives, and even aggregate statistics can
leak membership of a study; data access is therefore controlled and consent defines what may be done. Most GWAS
participants so far have been of European ancestry — roughly four in five in the survey of Martin *et al.* (2019) — so
both discoveries and scores serve some groups better than others. Polygenic scores are probabilistic risk
modifiers, not diagnoses; their clinical use (and their use for embryo selection, which professional societies
caution against) requires careful evaluation of calibration and benefit in each population. Results about behavioural
traits are especially easy to misread as deterministic.

## 📌 Key takeaways

- **Biobank GWAS = Module 06 at scale plus diagnostics.** Covariate-adjusted per-SNP regression (Frisch–Waugh–Lovell),
  Manhattan and QQ plots, $\lambda_{GC}$; Bonferroni ($5\times10^{-8}$) or FDR.
- **Population structure** creates genome-wide false positives when ancestry correlates with environment; genotype PCs
  (top eigenvectors of the GRM $K=ZZ^\top/J$) remove the broad axes of ancestry.
- **Linear mixed models** $y\sim\mathcal N(X\beta,\sigma_g^2K+\sigma_e^2I)$ estimate **SNP heritability** and correct for
  structure and relatedness. One eigendecomposition makes the likelihood a 1-D problem (EMMA, FaST-LMM); the LMM is ridge
  regression on all SNPs with the prior variance estimated by ML (BLUP, empirical Bayes).
- **Fine-mapping** turns towers of LD into probabilities: Bayes factors → PIPs → 95 % credible sets; SuSiE handles
  several causal variants by summing single effects. Calibration must be checked by simulation; perfect proxies cannot
  be separated statistically.
- **Polygenic scores** (C+T, ridge/BLUP, lasso, Bayesian shrinkage) are bounded by the heritability and lose accuracy
  in populations with different LD — even when the causal biology is identical.
- **Mendelian randomisation** estimates causal effects of exposures from SNP associations (Wald ratio, IVW) under three
  instrument assumptions; pleiotropy biases IVW, MR-Egger and the weighted median trade precision for robustness.
- **LD-score regression** separates polygenic signal (slope → $h^2$) from confounding (intercept) using summary
  statistics alone — under its own modelling assumptions.

---
## ✅ Checkpoint

1. Two people have $K_{ij}=0.5$ in a GRM built from standardised SNPs. What is their likely relationship, and what would you expect for two unrelated people genotyped at 5,000 independent SNPs?
   <details><summary>Answer</summary>About 0.5 is expected for full siblings or parent–offspring (twice the kinship coefficient 1/4). For unrelated people $K_{ij}$ scatters around 0 with standard deviation ≈ $1/\sqrt{5000}\approx0.014$, as in Section 3.1.</details>
2. A GWAS of 5,000 SNPs gives a median $\chi^2$ of 0.68. Compute $\lambda_{GC}$. Name two different explanations and one analysis that distinguishes them.
   <details><summary>Answer</summary>$\lambda_{GC}=0.68/0.455\approx1.49$. Confounding (population stratification, cryptic relatedness) or polygenic signal (many true small effects). LD-score regression separates them: an intercept near 1 with a positive slope means polygenicity; an intercept clearly above 1 means confounding.</details>
3. Why do both genotype PCs and a linear mixed model reduce inflation due to population structure?
   <details><summary>Answer</summary>Both model the genome-wide similarity of people: PCs as fixed-effect covariates along the top eigenvectors of the GRM, the LMM as a random effect with covariance $\sigma_g^2K$ that includes all eigenvectors (weighted by their eigenvalues). The LMM additionally handles fine-scale structure and relatedness.</details>
4. In the EMMA/FaST-LMM trick, why does a single eigendecomposition of $K$ suffice for every value of $\delta$?
   <details><summary>Answer</summary>$K+\delta I=U(S+\delta I)U^\top$ for the same $U$: adding $\delta I$ only shifts the eigenvalues. After rotating the data by $U^\top$ the covariance is diagonal for every $\delta$, so each likelihood evaluation costs $O(N)$.</details>
5. Show that a mixed model with $K=ZZ^\top/J$ is ridge regression. What is λ for $J=2{,}000$ standardised SNPs and $h^2=0.5$?
   <details><summary>Answer</summary>$g=Zu/\sqrt J$ with $u_k\sim\mathcal N(0,\sigma_g^2)$ gives $\operatorname{Cov}(g)=\sigma_g^2K$; the posterior mean of the per-SNP effects is ridge regression with $\lambda=J\sigma_e^2/\sigma_g^2=J(1-h^2)/h^2=2000$.</details>
6. Four SNPs in a region have Bayes factors 120, 60, 15 and 5 (single causal variant, uniform prior). Compute the PIPs and the 95 % credible set.
   <details><summary>Answer</summary>Sum = 200; PIPs 0.60, 0.30, 0.075, 0.025. Cumulative: 0.60, 0.90, 0.975 → the 95 % set is the first three SNPs.</details>
7. What does a 95 % credible set guarantee, and what does it not?
   <details><summary>Answer</summary>Under the model and prior, the causal variant is in the set with posterior probability ≥ 0.95, so about 95 % of sets contain it over studies where the model holds (checked by simulation). No guarantee when the model is wrong (several causal variants, LD mismatch); and it cannot shrink below a group of SNPs in perfect LD.</details>
8. Why does a polygenic score trained in one population lose accuracy in another, even if the causal effects are identical? State the three instrumental-variable assumptions of Mendelian randomisation.
   <details><summary>Answer</summary>The score uses tag SNPs whose correlation with the causal variants (LD) and whose allele frequencies differ between populations, so the weights no longer track the causal alleles (Section 5.5). IV assumptions: relevance (SNP associated with the exposure), independence (SNP independent of confounders), exclusion restriction (SNP affects the outcome only through the exposure).</details>

## 🧠 Questions & discussion

1. A twin study estimates the heritability of a trait at 0.7 and a GREML analysis of 20,000 unrelated people at 0.25. List three reasons for the gap, and say which of them a whole-genome-sequencing study could address.
2. Section 2's ✏️ showed that 'stratification' can be partly genetic. If the populations truly differ in the genetic basis of a trait, is correcting for PCs always the right thing to do? What is lost?
3. A polygenic score for coronary artery disease is offered to patients of all ancestries by a clinic. Using Section 5, write down what you would want to know about its calibration and accuracy in each group before using it — and who should decide.
4. An MR study finds that genetically higher BMI raises the risk of depression. List alternative explanations (pleiotropy, dynastic effects, selection into the biobank, social consequences of appearance) and an analysis that addresses each.
5. Fine-mapping narrows a locus to a credible set of 12 non-coding variants in perfect LD. What experiments or computational tools (Module 36) would you use to find the causal one?
6. Biobank participants consent to research use of their data. Should researchers be allowed to build polygenic predictors of educational attainment from such data? Discuss benefits, harms and safeguards.

## 🚀 Bonus — going further

- **Binary traits.** Threshold the trait (`y > np.quantile(y, 0.9)`), fit the LMM to the 0/1 outcome and transform $h^2$ to the liability scale; compare with the truth.
- **Conditional analysis vs SuSiE.** Add the lead SNP as a covariate and rescan while $p<5\times10^{-8}$; compare with SuSiE-lite over the regions of 4.8.
- **Colocalisation.** Simulate an eQTL and a GWAS trait that share (or not) a causal SNP; under single-causal assumptions the coloc idea is $\sum_j \mathrm{PIP}^{(1)}_j\mathrm{PIP}^{(2)}_j$.
- **Summary-statistic PGS.** Implement LDpred-inf, $\hat w=(R+\frac{M}{Nh^2}I)^{-1}\hat\beta_{marg}$, from the marginal effects of Section 5 with an LD matrix $R$ from another sample of population A, then of population B.
- Try the real tools on public summary statistics (outside this notebook): LDSC, susieR, TwoSampleMR.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 35 · Statistical genetics at scale
Biological question: which variants cause trait variation, how heritable is it, can we predict it portably,
                     and does an exposure cause an outcome?
Data (simulated): simulate_genotypes (1 and 3 populations, F_ST 0.05), simulate_ld_genotypes (LD blocks),
                  simulate_quantitative_trait (known causal SNPs, h²), in-notebook MR and LDSC simulators
Method: covariate-adjusted GWAS, PCs from the GRM, EMMA/FaST-LMM-style REML, EMMAX scan, single-effect regression,
        SuSiE-lite (IBSS), C+T / ridge-BLUP / lasso scores, IVW / MR-Egger / weighted median, LD-score regression
Key equation: y ~ N(Xβ, σ_g² K + σ_e² I);  PIP_j = ABF_j / Σ_k ABF_k;  θ_IVW = Σ βx βy σy⁻² / Σ βx² σy⁻²;
              E[χ²_j] = 1 + N h² ℓ_j / M + N a
Key parameters: number of PCs, h², prior variance W, credible level, L, p-threshold, λ, α, number of instruments
Ground-truth check result: λ_GC naive/PC/LMM = …; ĥ² = … (truth 0.5), CI coverage …; CS coverage …;
                           PGS R² A/B = …; IVW/Egger under pleiotropy = … (truth 0.3); LDSC intercept …
What changed when I modified … (n_causal / effect sizes / shared blocks / pleiotropy): …
One failure mode: (e.g. directional pleiotropy biases IVW; LD mismatch ruins portability)
Connection to research (paper/tool): GCTA / BOLT-LMM / SuSiE / PRS-CS / TwoSampleMR / LDSC
```

## 📚 References

- Price A. L., Patterson N. J., Plenge R. M., Weinblatt M. E., Shadick N. A., Reich D. (2006) Principal components analysis corrects for stratification in genome-wide association studies. *Nat Genet* 38:904–909.
- Yang J. *et al.* (2010) Common SNPs explain a large proportion of the heritability for human height. *Nat Genet* 42:565–569.
- Yang J., Lee S. H., Goddard M. E., Visscher P. M. (2011) GCTA: a tool for genome-wide complex trait analysis. *Am J Hum Genet* 88:76–82.
- Kang H. M. *et al.* (2010) Variance component model to account for sample structure in genome-wide association studies. *Nat Genet* 42:348–354.
- Kang H. M. *et al.* (2008) Efficient control of population structure in model organism association mapping. *Genetics* 178:1709–1723.
- Lippert C., Listgarten J., Liu Y., Kadie C. M., Davidson R. I., Heckerman D. (2011) FaST linear mixed models for genome-wide association studies. *Nat Methods* 8:833–835.
- Loh P.-R. *et al.* (2015) Efficient Bayesian mixed-model analysis increases association power in large cohorts. *Nat Genet* 47:284–290.
- Listgarten J., Lippert C., Kadie C. M., Davidson R. I., Eskin E., Heckerman D. (2012) Improved linear mixed models for genome-wide association studies. *Nat Methods* 9:525–526.
- Yang J., Zaitlen N. A., Goddard M. E., Visscher P. M., Price A. L. (2014) Advantages and pitfalls in the application of mixed-model association methods. *Nat Genet* 46:100–106.
- Visscher P. M. *et al.* (2014) Statistical power to detect genetic (co)variance of complex traits using SNP data in unrelated samples. *PLoS Genet* 10:e1004269.
- Haseman J. K., Elston R. C. (1972) The investigation of linkage between a quantitative trait and a marker locus. *Behav Genet* 2:3–19.
- Bulik-Sullivan B. K. *et al.* (2015) LD Score regression distinguishes confounding from polygenicity in genome-wide association studies. *Nat Genet* 47:291–295.
- Li N., Stephens M. (2003) Modeling linkage disequilibrium and identifying recombination hotspots using single-nucleotide polymorphism data. *Genetics* 165:2213–2233.
- Maller J. B. *et al.* (2012) Bayesian refinement of association signals for 14 loci in 3 common diseases. *Nat Genet* 44:1294–1301.
- Wakefield J. (2009) Bayes factors for genome-wide association studies: comparison with P-values. *Genet Epidemiol* 33:79–86.
- Schaid D. J., Chen W., Larson N. B. (2018) From genome-wide associations to candidate causal variants by statistical fine-mapping. *Nat Rev Genet* 19:491–504.
- Wang G., Sarkar A., Carbonetto P., Stephens M. (2020) A simple new approach to variable selection in regression, with application to genetic fine mapping. *J R Stat Soc B* 82:1273–1300.
- Daetwyler H. D., Villanueva B., Woolliams J. A. (2008) Accuracy of predicting the genetic risk of disease using a genome-wide approach. *PLoS One* 3:e3395.
- Purcell S. M. *et al.* (2009) Common polygenic variation contributes to risk of schizophrenia and bipolar disorder. *Nature* 460:748–752.
- Choi S. W., Mak T. S.-H., O'Reilly P. F. (2020) Tutorial: a guide to performing polygenic risk score analyses. *Nat Protoc* 15:2759–2772.
- Martin A. R., Kanai M., Kamatani Y., Okada Y., Neale B. M., Daly M. J. (2019) Clinical use of current polygenic risk scores may exacerbate health disparities. *Nat Genet* 51:584–591.
- Davey Smith G., Ebrahim S. (2003) 'Mendelian randomization': can genetic epidemiology contribute to understanding environmental determinants of disease? *Int J Epidemiol* 32:1–22.
- Burgess S., Butterworth A., Thompson S. G. (2013) Mendelian randomization analysis with multiple genetic variants using summarized data. *Genet Epidemiol* 37:658–665.
- Bowden J., Davey Smith G., Burgess S. (2015) Mendelian randomization with invalid instruments: effect estimation and bias detection through Egger regression. *Int J Epidemiol* 44:512–525.
- Bowden J., Davey Smith G., Haycock P. C., Burgess S. (2016) Consistent estimation in Mendelian randomization with some invalid instruments using a weighted median estimator. *Genet Epidemiol* 40:304–314.
- Voight B. F. *et al.* (2012) Plasma HDL cholesterol and risk of myocardial infarction: a mendelian randomisation study. *Lancet* 380:572–580.
- Bycroft C. *et al.* (2018) The UK Biobank resource with deep phenotyping and genomic data. *Nature* 562:203–209.
- Visscher P. M. *et al.* (2017) 10 years of GWAS discovery: biology, function, and translation. *Am J Hum Genet* 101:5–22.
- Uffelmann E. *et al.* (2021) Genome-wide association studies. *Nat Rev Methods Primers* 1:59.

## ➡️ Next

Fine-mapping left us with credible sets of mostly **non-coding** variants. Which of them changes how a gene is
regulated — and how? **[Module 36 · Regulatory genomics: deep learning from DNA sequence to function](36_regulatory_genomics_deep_learning.ipynb)**
trains a convolutional network on a simulated reporter assay, opens it up to find transcription-factor motifs and
their grammar, and predicts the effect of every single-letter change.